# 머신러닝-2 · 전처리 · 누수 · 파이프라인

**한 줄 요약** · 글자 · 단위 · 빈칸을 모델 입력으로 바꾸되, 그 기준은 train으로만 — Pipeline 하나로 지킴

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. Label · Ordinal · OneHot · `get_dummies` 중 상황에 맞는 것을 고르고, 번호가 만드는 가짜 순서(0.640)를 설명한다
2. Standard · MinMax · Robust의 차이를 이상치 예로 말하고, KNN이 스케일링으로 0.784 → 0.982가 됨을 보인다
3. `fit`은 train에만 · `transform`은 전부라는 규칙을 지키고, 누수가 없는 패턴(0.870)을 만드는 이유를 말한다
4. `SimpleImputer` · `ColumnTransformer` · `Pipeline`으로 타이타닉 전처리를 묶고 `이름__설정`으로 탐색한다
5. 이미지 mean · std 정규화와 범주 → 번호 → 임베딩이 이 장의 무엇과 같은 원리인지 설명한다

## 어디에 쓰나

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">실제 상황</th>
<th style="text-align: left;">이 권에서 배우는 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">표에 성별 · 지역 같은 글자 열이 있을 때</td>
<td style="text-align: left;">원-핫 · 순서 지정 · 처음 보는 값 처리 (1장)</td>
</tr>
<tr>
<td style="text-align: left;">금액 · 나이처럼 단위가 다른 열을 KNN · 로지스틱에 넣을 때</td>
<td style="text-align: left;">스케일러 고르기 (2장 · 선택표)</td>
</tr>
<tr>
<td style="text-align: left;">교차검증 점수가 이상하게 높을 때</td>
<td style="text-align: left;">누수 점검 · fit 범위 (3장)</td>
</tr>
<tr>
<td style="text-align: left;">결측 · 글자 · 숫자가 섞인 표를 매번 같은 방식으로 처리할 때</td>
<td style="text-align: left;">Pipeline · ColumnTransformer (4장)</td>
</tr>
<tr>
<td style="text-align: left;">이미지 정규화 값 · 단어 사전을 정할 때</td>
<td style="text-align: left;">train 통계 · <code>&lt;unk&gt;</code> · 임베딩 (5장)</td>
</tr>
</tbody>
</table>

## 학습 지도

<img src="images/ml2_map.png" alt="머신러닝-2 학습 지도" width="837">

**예상 학습 시간** · 개념 2시간 + 실습 1시간 (전체 실행 1분 이내 · GPU 필요 없음 · 5-2만 PyTorch 사용)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: center;">장</th>
<th style="text-align: left;">제목</th>
<th style="text-align: left;">이 장의 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: center;">0</td>
<td style="text-align: left;">준비</td>
<td style="text-align: left;">라이브러리 · seed · 5등분 규칙 · 타이타닉</td>
</tr>
<tr>
<td style="text-align: center;">1</td>
<td style="text-align: left;">인코딩</td>
<td style="text-align: left;">글자를 숫자로 바꾸는 4가지는 어떻게 다르고, 번호는 무엇을 왜곡하나?</td>
</tr>
<tr>
<td style="text-align: center;">2</td>
<td style="text-align: left;">스케일링 · 정규화</td>
<td style="text-align: left;">단위를 어떻게 맞추고, 스케일러 3종 · 정규화는 언제 쓰나?</td>
</tr>
<tr>
<td style="text-align: center;">3</td>
<td style="text-align: left;">fit · transform · 누수</td>
<td style="text-align: left;">기준은 어느 데이터로 만들고, 어기면 평가가 왜 무효인가?</td>
</tr>
<tr>
<td style="text-align: center;">4</td>
<td style="text-align: left;">파이프라인</td>
<td style="text-align: left;">결측 · 인코딩 · 스케일링 · 모델을 하나로 묶어 탐색하려면?</td>
</tr>
<tr>
<td style="text-align: center;">5</td>
<td style="text-align: left;">딥러닝 연결</td>
<td style="text-align: left;">이미지 정규화 · 임베딩은 이 장의 무엇과 같은가?</td>
</tr>
<tr>
<td style="text-align: center;">6</td>
<td style="text-align: left;">종합 실습</td>
<td style="text-align: left;">모델별 전처리를 누수 없이 비교하려면?</td>
</tr>
<tr>
<td style="text-align: center;">—</td>
<td style="text-align: left;">마무리 · 부록</td>
<td style="text-align: left;">오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td>
</tr>
</tbody>
</table>

---
## 0. 준비

모든 장에서 쓰는 라이브러리 · seed · 5등분 규칙과 타이타닉 데이터를 불러옴.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">데이터</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">처음 나오는 곳</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>titanic</code></td>
<td style="text-align: left;">승객 891명 × 12열 (<code>train.csv</code>, 정답 <code>Survived</code>) · 결측 Age 177 · Cabin 687 · Embarked 2</td>
<td style="text-align: left;">0</td>
</tr>
<tr>
<td style="text-align: left;">색 · 크기 · 문장</td>
<td style="text-align: left;">원본 노트북의 작은 예제 (코드 안에서 생성)</td>
<td style="text-align: left;">1장</td>
</tr>
<tr>
<td style="text-align: left;"><code>Xp_tr</code> · <code>yp_tr</code></td>
<td style="text-align: left;">펭귄 train 273마리 (머신러닝-1 6장과 같은 분할)</td>
<td style="text-align: left;">2-3</td>
</tr>
<tr>
<td style="text-align: left;"><code>X_iris</code></td>
<td style="text-align: left;">붓꽃 150송이 (원본 Part 4와 같은 7 : 3 분할)</td>
<td style="text-align: left;">3-1</td>
</tr>
<tr>
<td style="text-align: left;"><code>X_train</code> · <code>X_valid</code> · <code>X_test</code></td>
<td style="text-align: left;">타이타닉 534 · 178 · 179명 (test 봉인)</td>
<td style="text-align: left;">4-2</td>
</tr>
<tr>
<td style="text-align: left;"><code>cv5</code></td>
<td style="text-align: left;">섞은 5등분 규칙 (seed 42)</td>
<td style="text-align: left;">0</td>
</tr>
</tbody>
</table>

**실행 흐름**

1. 라이브러리 불러오기
2. seed · 5등분 규칙 만들기 · 버전 기록
3. 타이타닉 `train.csv` 읽기
4. 열마다 빈칸 수 확인

In [1]:
# 라이브러리 불러오기
import sklearn                                       # 버전 기록용
import numpy as np                                   # 배열 계산
import pandas as pd                                  # 표 데이터
import seaborn as sns                                # 펭귄 데이터 (머신러닝-1 6장과 같은 방법)
from sklearn.linear_model import LogisticRegression  # 선형(가중합) 모델
from sklearn.tree import DecisionTreeClassifier      # 트리 모델 (스케일 영향 없음)
from sklearn.neighbors import KNeighborsClassifier   # 거리 모델 (스케일 영향 큼)
from sklearn.model_selection import (                # 분할 · 교차검증 (머신러닝-1)
    train_test_split, StratifiedKFold, cross_val_score)


# seed (원본 전처리 노트북과 같은 42) · 5등분 규칙 (머신러닝-1 3-1)
SEED = 42                                                           # 원본 전처리 노트북과 같은 seed
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)  # 모든 비교에 같은 5등분
print("sklearn:", sklearn.__version__, "| pandas:", pd.__version__)  # 버전 기록


# 타이타닉 (4장 · 6장): 노트북과 같은 폴더의 train.csv
titanic = pd.read_csv("train.csv")       # 노트북 옆 파일
print("타이타닉:", titanic.shape,        # 크기 · 생존 비율
      "| 생존 비율:", round(titanic["Survived"].mean(), 3))
na = titanic.isna().sum()                # 열마다 빈칸 수
print("결측 수:", na[na > 0].to_dict())  # 결측이 있는 열만

sklearn: 1.9.0 | pandas: 3.0.3
타이타닉: (891, 12) | 생존 비율: 0.384
결측 수: {'Age': 177, 'Cabin': 687, 'Embarked': 2}


---
## 1. 인코딩

> **이 장의 흐름**
>
> - 머신러닝-1에서는 이미 숫자인 붓꽃 · 펭귄만 썼다
> - 실제 표에는 성별 · 항구 · 문장 같은 글자가 섞여 있고 모델은 숫자만 읽음.
> - 번역 방법에 따라 모델이 받는 정보가 달라짐

**파이프라인 위치** · ③ 전처리 — 범주 · 텍스트 → 숫자

### 1-1. `A` 글자를 숫자로 바꾸는 3가지 · `LabelEncoder` · `OneHotEncoder` · `get_dummies` <sub>원본 03 Part 1 셀 2 ~ 7</sub>

> 🎯 **글자(범주)를 모델이 계산할 수 있는 숫자로 바꾸는 방법은 무엇이 있고, 나중에 들어온 데이터에서는 어떻게 다른가?**

💡 **쉽게 말하면**

- 모델은 **숫자만 읽는 계산기**.
- 색 이름을 번호 하나로 바꾸거나(Label), 색마다 스위치 칸을 하나씩 만들어 켜고 끄는(One-Hot) 방식으로 번역함.

<img src="images/ml2_encoding.png" alt="번호 1열 vs 원-핫 3열 · 처음 보는 값" width="763">

**실행 흐름**

1. 색 6개
2. LabelEncoder 번호
3. OneHotEncoder 0/1 열
4. get_dummies
5. 처음 보는 색(purple)으로 차이 확인

**먼저 예측** · train에 없던 `purple`이 들어오면 `OneHotEncoder`와 `get_dummies`의 열은 어떻게 될까? — 답은 코드 실행 결과 아래에

In [2]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder  # 인코더 2종


# 원본 셀: 색깔 6개
data = pd.DataFrame({"color": ["red", "green", "blue",  # 원본과 같은 색 6개
                               "red", "green", "blue"]})


# 방법 1: LabelEncoder → 열 1개에 번호 (알파벳 순서로 0 · 1 · 2)
le = LabelEncoder()                                              # 번호 붙이는 도구
data["label"] = le.fit_transform(data["color"])                  # 범주 기억 + 번호로
print("Label 번호:", dict(zip(le.classes_.tolist(), range(3))))  # 어떤 번호가 붙었나


# 방법 2: OneHotEncoder → 범주마다 0/1 열 (입력은 2차원 [[ ]])
ohe = OneHotEncoder(sparse_output=False)                   # 결과를 일반 배열로
onehot = ohe.fit_transform(data[["color"]]).astype(int)    # 범주마다 0/1 열
print("OneHot 열:", ohe.get_feature_names_out().tolist())  # 새로 생긴 열 이름
print("red 행:", onehot[0], "| green 행:", onehot[1])      # 자기 칸만 1


# 방법 3: pd.get_dummies → 같은 0/1 표를 pandas에서 바로
dummies = pd.get_dummies(data["color"], dtype=int)  # 같은 표를 pandas로
print("get_dummies 열:", dummies.columns.tolist())  # 열 이름 확인


# 차이: 나중에 들어온 데이터에 처음 보는 범주(purple)가 있으면
new = pd.DataFrame({"color": ["green", "purple"]})                           # 나중에 들어온 데이터
ohe_safe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")       # 모르는 값은 전부 0
ohe_safe.fit(data[["color"]])  # 기준 = 처음 데이터의 범주 3개
print("OneHot (기준 유지):", ohe_safe.transform(new[["color"]]).tolist())    # 열 3개 유지
print("get_dummies (새로):", pd.get_dummies(new["color"]).columns.tolist())  # 열이 바뀜

Label 번호: {'blue': 0, 'green': 1, 'red': 2}
OneHot 열: ['color_blue', 'color_green', 'color_red']
red 행: [0 0 1] | green 행: [0 1 0]
get_dummies 열: ['blue', 'green', 'red']
OneHot (기준 유지): [[0.0, 1.0, 0.0], [0.0, 0.0, 0.0]]
get_dummies (새로): ['green', 'purple']


**결과** · 예측 정답은 `OneHotEncoder`는 기존 3열 그대로(전부 0), `get_dummies`는 `['green', 'purple']`로 **열 자체가 바뀜**

- `Label 번호: {'blue': 0, 'green': 1, 'red': 2}` — 알파벳 순서로 번호 → 크기 순서가 생김 (1-2의 문제)
- `OneHot 열: ['color_blue', 'color_green', 'color_red']` — 범주 3개 → 열 3개 (행 수는 그대로, 열만 늘어남)
- `red 행: [0 0 1] | green 행: [0 1 0]` — 자기 칸만 1, 순서 없음
- `get_dummies 열: ['blue', 'green', 'red']` — 같은 표를 pandas에서 바로
- `OneHot (기준 유지): [[0.0, 1.0, 0.0], [0.0, 0.0, 0.0]]` — purple은 모르는 값 → 전부 0, 열 3개 유지 (`handle_unknown="ignore"`)
- `get_dummies (새로): ['green', 'purple']` — 새 데이터로 다시 만들어 열 이름 · 개수가 달라짐 → 모델 입력과 어긋남

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>LabelEncoder</code></td>
<td style="text-align: left;">열 1개에 번호 (알파벳 순서), 입력은 1차원</td>
<td style="text-align: left;">정답 <strong>y</strong>를 번호로 (1-3)</td>
</tr>
<tr>
<td style="text-align: left;"><code>OrdinalEncoder</code></td>
<td style="text-align: left;">열 1개에 번호, <strong>순서를 직접 지정</strong> 가능, 입력은 2차원</td>
<td style="text-align: left;">크기 · 등급처럼 진짜 순서가 있는 X (1-3)</td>
</tr>
<tr>
<td style="text-align: left;"><code>OneHotEncoder</code></td>
<td style="text-align: left;">범주마다 0/1 열, <strong>fit한 범주를 기억</strong> → 새 데이터도 같은 열</td>
<td style="text-align: left;">순서 없는 X (기본 선택, Pipeline 안에서)</td>
</tr>
<tr>
<td style="text-align: left;"><code>pd.get_dummies</code></td>
<td style="text-align: left;">같은 0/1 표를 바로 만들지만 <strong>기준을 기억하지 않음</strong> → 데이터마다 열이 달라짐</td>
<td style="text-align: left;">한 번 보고 끝나는 탐색 · 시각화</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `OneHotEncoder`는 2차원 입력 `df[["color"]]` (대괄호 2겹), `LabelEncoder`는 1차원 `df["color"]`

> ✅ **한 줄 정리** · 모델 입력에는 범주를 기억하는 `OneHotEncoder`를 쓰고, `get_dummies`는 한 번 보고 끝나는 탐색에만 쓴다.

**딥러닝 코드에서는**

- 텍스트 · 범주 사전도 같은 원리
- train으로 만든 사전을 저장해 두고 valid · test · 실제 입력에 그대로 씀 (5-2)

### 1-2. `A` 번호의 가짜 순서 · `LabelEncoder + 선형 모델` <sub>원본 03 셀 8 ~ 14</sub>

> 🎯 **범주에 0 · 1 · 2 번호만 붙여 선형 모델에 넣으면 모델은 그 번호를 어떻게 이해하는가?**

💡 **쉽게 말하면**

- 번호표는 **이름표**인데, 로지스틱은 그것을 **눈금자 위의 길이**로 읽음.
- blue(0)보다 red(2)가 '두 배 더 무엇'이라고 믿어 버림.

**실행 흐름**

1. 색 100개 (red · blue → 1, green → 0)
2. Label · OneHot으로 로지스틱
3. Label 모델의 번호별 확률
4. 트리로 다시

**먼저 예측** · red · blue가 1, green(가운데 번호 1)이 0인 규칙을 Label 번호 하나로 로지스틱이 표현할 수 있을까? — 답은 코드 실행 결과 아래에

In [3]:
# 원본 셀: 색 100개, red · blue → 1, green → 0 (번호 순서와 무관한 규칙)
np.random.seed(42)                                                        # 원본과 같은 색 100개
colors = pd.Series(np.random.choice(["red", "green", "blue"], size=100))  # 무작위 색
y_c = colors.map({"red": 1, "blue": 1, "green": 0})                       # 가운데 번호만 0인 규칙


# 같은 정보를 두 가지로 표현
X_label = LabelEncoder().fit_transform(colors).reshape(-1, 1)  # 1열
X_onehot = OneHotEncoder(sparse_output=False).fit_transform(   # 3열 표현
    colors.to_frame())                                         # 3열


# 같은 로지스틱에 두 표현 → 학습 데이터 정확도 (규칙을 표현할 수 있는가)
for name, Xc in [("Label", X_label), ("OneHot", X_onehot)]:  # 두 표현마다
    acc = LogisticRegression().fit(Xc, y_c).score(Xc, y_c)   # 학습 데이터 점수 = 표현 가능?
    print(f"로지스틱 · {name:<6}: {acc:.3f}")                # 표현별 점수
print(f"다수 클래스(1) 비율 : {y_c.mean():.3f}")             # 전부 1로 찍은 점수


# Label 모델을 열어 보기: 번호가 커질수록 확률도 커짐 (번호를 크기로 믿음)
lr_label = LogisticRegression().fit(X_label, y_c)    # Label 모델을 열어 봄
for v, name in enumerate(["blue", "green", "red"]):  # 번호 0 · 1 · 2마다
    p = lr_label.predict_proba([[v]])[0, 1]          # [0, 1] → 1일 확률
    print(f"{name:<5}({v}) → P(1) = {p:.4f}")        # 번호가 클수록 확률도 큼


# 트리는 번호 사이를 여러 번 잘라 두 표현 모두 맞힘
for name, Xc in [("Label", X_label), ("OneHot", X_onehot)]:      # 두 표현마다
    tree = DecisionTreeClassifier(random_state=42).fit(Xc, y_c)  # 트리로 다시
    print(f"트리 · {name:<6}: {tree.score(Xc, y_c):.3f}")        # 트리는 둘 다 맞힘

로지스틱 · Label : 0.640
로지스틱 · OneHot: 1.000
다수 클래스(1) 비율 : 0.640
blue (0) → P(1) = 0.6292
green(1) → P(1) = 0.6398
red  (2) → P(1) = 0.6503
트리 · Label : 1.000
트리 · OneHot: 1.000


**결과** · 예측 정답은 못 함 — `0.640` = 전부 1로 찍은 것과 같음 (가운데만 0인 규칙은 직선 하나로 표현 불가)

- `로지스틱 · Label : 0.640` — 다수 클래스 비율과 같음 → 모든 색에 1을 예측
- `로지스틱 · OneHot: 1.000` — 색마다 가중치를 따로 받아 규칙을 그대로 표현
- `다수 클래스(1) 비율 : 0.640` — 1이 64개, 0이 36개
- `blue (0) → P(1) = 0.6292` — 번호가 커질수록 확률이 커짐
- `red  (2) → P(1) = 0.6503` — 알파벳 순 번호를 진짜 크기로 믿음 → 셋 다 0.5 이상이라 전부 1
- `트리 · Label : 1.000` — 트리는 '번호 < 0.5', '번호 < 1.5'처럼 여러 번 잘라 표현 가능

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">번호(Label) + 선형 모델</td>
<td style="text-align: left;">번호를 <strong>크기</strong>로 믿음 → 순서가 없는 범주면 왜곡</td>
<td style="text-align: left;">쓰지 않음 (순서 없는 X일 때)</td>
</tr>
<tr>
<td style="text-align: left;">원-핫 + 선형 모델</td>
<td style="text-align: left;">범주마다 따로 가중치 → 순서 왜곡 없음</td>
<td style="text-align: left;">로지스틱 · 선형 회귀 · KNN</td>
</tr>
<tr>
<td style="text-align: left;">번호 + 트리</td>
<td style="text-align: left;">번호 사이를 여러 번 잘라 표현 가능하지만 범주가 많으면 잘라야 할 곳이 많아짐</td>
<td style="text-align: left;">범주가 많아 원-핫 열이 너무 늘 때의 대안</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 학습 데이터 점수라서 성능이 아니라 **표현 가능 여부**만 본 것
> - 트리라고 번호가 항상 안전하지도 않음 (범주가 많으면 잘라야 할 곳도 많아짐)

> ✅ **한 줄 정리** · 순서가 없는 범주에 번호를 붙이면 선형 모델은 그 번호를 크기로 믿으므로, 순서 없는 X는 원-핫으로 바꾼다.

**딥러닝 코드에서는**

- 딥러닝에서 범주 번호를 그대로 입력값으로 넣으면 같은 문제 → 번호는 `nn.Embedding`의 **찾아보기 번호**로만 씀 (5-2)

### 1-3. `A` 순서가 있는 범주 · `OrdinalEncoder` <sub>원본 03 셀 15 ~ 18</sub>

> 🎯 **small · medium · large처럼 진짜 순서가 있는 범주는 어떻게 번호를 붙이며, `LabelEncoder`와 무엇이 다른가?**

💡 **쉽게 말하면**

- 티셔츠 크기는 **진짜 줄 세우기**가 가능함.
- 그 순서를 내가 직접 알려 주는 것이 Ordinal, 알파벳순으로 아무렇게나 세우는 것이 Label.

**실행 흐름**

1. 크기 5개
2. OrdinalEncoder(순서 지정)
3. LabelEncoder(알파벳) 비교
4. LabelEncoder로 정답 y 번호 ↔ 이름

**먼저 예측** · `LabelEncoder`로 small · medium · large를 바꾸면 small의 번호는? — 답은 코드 실행 결과 아래에

In [4]:
from sklearn.preprocessing import OrdinalEncoder  # 순서를 지정하는 인코더


# 원본 셀: 크기에는 진짜 순서가 있음 (small < medium < large)
sizes = pd.DataFrame({"size": ["small", "medium", "large",  # 진짜 순서가 있는 범주
                               "medium", "small"]})


# OrdinalEncoder: 순서를 직접 지정 → small 0 · medium 1 · large 2
order = [["small", "medium", "large"]]  # 열마다 순서 목록 → 리스트 안 리스트
sizes["ordinal"] = OrdinalEncoder(categories=order).fit_transform(  # 지정 순서대로 번호
    sizes[["size"]]).ravel()            # (5, 1) → (5,)


# 비교: LabelEncoder는 알파벳 순서 → large 0 · medium 1 · small 2
sizes["label"] = LabelEncoder().fit_transform(sizes["size"])  # 알파벳 순 번호 (비교)
print(sizes.to_string(index=False))                           # 두 번호 나란히


# LabelEncoder의 원래 용도 = 정답 y (이름 → 번호 → 다시 이름)
le_y = LabelEncoder().fit(["Adelie", "Gentoo", "Chinstrap", "Adelie"])  # 정답 이름 기억
codes = le_y.transform(["Gentoo", "Adelie"])                            # 이름 → 번호
print("y 번호:", codes, "→ 되돌리기:", le_y.inverse_transform(codes))   # 번호 → 이름

  size  ordinal  label
 small      0.0      2
medium      1.0      1
 large      2.0      0
medium      1.0      1
 small      0.0      2
y 번호: [2 0] → 되돌리기: ['Gentoo' 'Adelie']


**결과** · 예측 정답은 `2` — 알파벳 순서(large 0 · medium 1 · small 2)라 크기 순서와 거꾸로

- `small      0.0      2` — Ordinal 0 (가장 작음) vs Label 2 (알파벳 마지막)
- `large      2.0      0` — Ordinal은 순서 그대로, Label은 뒤집힘
- `y 번호: [2 0] → 되돌리기: ['Gentoo' 'Adelie']` — 정답 y는 번호 ↔ 이름 왕복만 되면 충분 → Label의 원래 용도

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>OrdinalEncoder(categories=[[...]])</code></td>
<td style="text-align: left;">내가 정한 <strong>의미 있는 순서</strong></td>
<td style="text-align: left;">small &lt; medium &lt; large, 등급 · 학년</td>
</tr>
<tr>
<td style="text-align: left;"><code>LabelEncoder</code></td>
<td style="text-align: left;"><strong>알파벳 순서</strong> (large 0 · medium 1 · small 2)</td>
<td style="text-align: left;">정답 y (순서 의미 없음, <code>inverse_transform</code>으로 되돌리기)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `categories=[[...]]`는 **리스트 안 리스트** (열마다 하나), 오타가 있는 범주는 fit에서 에러

> ✅ **한 줄 정리** · 진짜 순서가 있는 X는 `OrdinalEncoder(categories=...)`로 순서를 지정하고, `LabelEncoder`는 정답 y에 쓴다.

**딥러닝 코드에서는**

- 분류 정답 y는 `LabelEncoder` 번호가 그대로 `CrossEntropyLoss`의 정답 번호
- `classes_` 순서를 저장해 두어야 예측 번호를 이름으로 되돌림

### 1-4. `B` 문장을 숫자로 · `CountVectorizer` <sub>원본 03 셀 19 ~ 26</sub>

> 🎯 **문장은 어떻게 숫자 벡터가 되고, 사전에 없는 단어만 있는 문장은 어떻게 처리되는가?**

💡 **쉽게 말하면**

- `CountVectorizer`는 **단어장 + 출석부**.
- fit에서 단어장을 만들고, 새 문장은 단어장에 있는 단어만 출석을 부름.

**실행 흐름**

1. 문장 6개 fit
2. 사전 14개
3. 로지스틱 학습
4. 사전에 없는 문장 · 있는 문장 transform
5. 확률 · 예측

**먼저 예측** · "이 영화는 별로다"는 사전 단어가 몇 개 맞을까? (`영화는` ≠ `영화`, `별로다` ≠ `별로였다`) — 답은 코드 실행 결과 아래에

In [5]:
from sklearn.feature_extraction.text import CountVectorizer  # 문장 → 단어 개수


# 원본 셀: 문장 6개 · 1 = 긍정, 0 = 부정
texts = ["이 영화 정말 재밌다", "완전 감동적이다", "너무 지루하고 별로였다",  # 원본 문장 6개
         "시간 낭비였다", "연출이 좋았다", "스토리가 엉망이다"]
labels = [1, 1, 0, 0, 1, 0]                                                   # 1 = 긍정, 0 = 부정


# fit → 단어 사전 만들기 (한 글자 '이'는 기본 설정에서 빠짐)
vec = CountVectorizer()                             # 단어장 만드는 도구
X_text = vec.fit_transform(texts)                   # (문장 수, 사전 단어 수) 개수 표
words = vec.get_feature_names_out()                 # 사전 단어 목록
print("사전 단어 수:", len(words), "| 앞 5개:", words[:5].tolist())  # 사전 크기
model_t = LogisticRegression().fit(X_text, labels)  # 개수 벡터로 학습


# 새 문장 → transform만 (fit 하면 사전이 바뀜)
for sent in ["이 영화는 별로다", "정말 지루하고 별로였다"]:  # 새 문장 2개
    x = vec.transform([sent])                                # 기존 사전으로만 세기
    hit = words[x.toarray()[0] > 0].tolist()                 # 사전에 있는 단어
    proba = model_t.predict_proba(x)[0].round(4)             # [부정, 긍정]
    print(f"{sent} → 사전 단어 {hit} · 확률 {proba}"         # 읽은 단어 · 확률 · 예측
          f" · 예측 {model_t.predict(x)[0]}")

사전 단어 수: 14 | 앞 5개: ['감동적이다', '낭비였다', '너무', '별로였다', '스토리가']
이 영화는 별로다 → 사전 단어 [] · 확률 [0.5 0.5] · 예측 1
정말 지루하고 별로였다 → 사전 단어 ['별로였다', '정말', '지루하고'] · 확률 [0.5728 0.4272] · 예측 0


**결과** · 예측 정답은 `0`개 — 확률 `[0.5 0.5]`, 즉 근거 없는 반반

- `사전 단어 수: 14 | 앞 5개: ['감동적이다', '낭비였다', '너무', '별로였다', '스토리가']` — 6문장의 단어 14개 (한 글자 '이'는 기본 설정에서 빠짐)
- `이 영화는 별로다 → 사전 단어 [] · 확률 [0.5 0.5] · 예측 1` — 아무 단어도 못 읽음 → 반반인데도 답은 나옴 (원본 출력은 0, 반반이라 계산 오차로 갈림)
- `정말 지루하고 별로였다 → 사전 단어 ['별로였다', '정말', '지루하고'] · 확률 [0.5728 0.4272] · 예측 0` — 사전 단어 3개 → 부정 쪽 근거가 생김

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>vec.fit_transform(train 문장)</code></td>
<td style="text-align: left;">단어 사전 만들기 + 개수 세기</td>
<td style="text-align: left;">학습 문장에 한 번</td>
</tr>
<tr>
<td style="text-align: left;"><code>vec.transform(새 문장)</code></td>
<td style="text-align: left;"><strong>기존 사전</strong>으로만 세기, 없는 단어는 버림</td>
<td style="text-align: left;">검증 · 테스트 · 실제 입력 (항상)</td>
</tr>
<tr>
<td style="text-align: left;">새 문장에 <code>fit</code></td>
<td style="text-align: left;">사전이 바뀌어 모델이 배운 열과 어긋남</td>
<td style="text-align: left;">쓰지 않음</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 출력이 나왔다고 근거가 있는 것은 아님
> - 확률이 0.5 근처면 예측 번호보다 확률을 먼저 확인

> ✅ **한 줄 정리** · 문장은 train으로 만든 단어 사전으로 개수를 세어 벡터로 바꾸고, 새 문장은 `transform`만 한다.

**딥러닝 코드에서는**

- 딥러닝 텍스트 모델도 train으로 단어(토큰) 사전을 만들고 사전에 없는 단어는 `<unk>` 번호로 보냄 (5-2)
- 한국어는 형태소 · 서브워드 토크나이저로 `영화는`을 `영화 + 는`으로 쪼갬

---
## 2. 스케일링 · 정규화

> **이 장의 흐름**
>
> - 1장에서 모든 열이 숫자가 됐다
> - 그런데 운임(수백)과 형제 수(0~8)처럼 단위가 다르면 거리 · 가중합에서 큰 단위가 이김.
> - 열마다 눈금을 맞추고, 행마다 길이를 맞추는 법

**파이프라인 위치** · ③ 전처리 — 단위 맞추기 · 방향만 남기기

### 2-1. `A` 단위가 다른 특성 맞추기 · `StandardScaler` <sub>원본 03 셀 27 ~ 30 · 미션 3 정답1 거리 몫</sub>

> 🎯 **단위가 다른 특성은 거리 계산에서 어떤 문제를 만들고, `StandardScaler`는 무엇을 계산하는가?**

💡 **쉽게 말하면**

- 키는 cm, 용돈은 원이면 **용돈 1원 차이와 키 1cm 차이가 같은 무게**로 계산됨.
- 그래서 열마다 '평균에서 표준편차 몇 칸 떨어졌나'로 단위를 지움.

**실행 흐름**

1. 키 · 용돈 3명
2. fit(평균 · 표준편차)
3. transform
4. 공식과 비교
5. 학생 0 ↔ 1 거리에서 특성별 몫

**먼저 예측** · 스케일링 전, 학생 0과 1 사이 거리에서 키가 차지하는 몫은? — 답은 코드 실행 결과 아래에

In [6]:
from sklearn.preprocessing import StandardScaler  # 평균 0 · 표준편차 1


# 원본 셀: 키(cm) · 용돈(원) → 범위가 20 vs 49000
X_hm = np.array([[150, 1000], [160, 20000], [170, 50000]], dtype=float)  # 키 · 용돈 3명


# fit = 열마다 평균 · 표준편차 계산, transform = (값 - 평균) / 표준편차
scaler = StandardScaler().fit(X_hm)                     # 열마다 기준 계산
X_hm_sc = scaler.transform(X_hm)                        # 기준으로 변환
print("평균:", scaler.mean_.round(1).tolist(),          # 저장된 기준
      "| 표준편차:", scaler.scale_.round(1).tolist())
print("키 변환:", X_hm_sc[:, 0].round(3), "| 용돈 변환:", X_hm_sc[:, 1].round(3))  # 같은 눈금
manual = (X_hm - X_hm.mean(axis=0)) / X_hm.std(axis=0)  # 공식으로 직접
print("공식과 같음:", np.allclose(X_hm_sc, manual))     # 공식 확인


# 학생 0 ↔ 1 거리에서 각 특성의 몫 (미션 3의 '거리 몫'과 같은 계산)
for name, M in [("전", X_hm), ("후", X_hm_sc)]:  # 변환 전 · 후
    d2 = (M[0] - M[1]) ** 2                      # 특성별 차이의 제곱
    share = d2 / d2.sum()                        # 거리에서 차지하는 비율
    print(f"스케일링 {name}: 키 몫 {share[0]:.1%} · 용돈 몫 {share[1]:.1%}")  # 몫 비교

평균: [160.0, 23666.7] | 표준편차: [8.2, 20171.5]
키 변환: [-1.225  0.     1.225] | 용돈 변환: [-1.124 -0.182  1.305]
공식과 같음: True
스케일링 전: 키 몫 0.0% · 용돈 몫 100.0%
스케일링 후: 키 몫 62.8% · 용돈 몫 37.2%


**결과** · 예측 정답은 `0.0%` — 키 차이(10)의 제곱은 용돈 차이(19000)의 제곱에 비해 무시됨

- `평균: [160.0, 23666.7] | 표준편차: [8.2, 20171.5]` — fit이 열마다 계산해 저장한 기준
- `키 변환: [-1.225  0.     1.225] | 용돈 변환: [-1.124 -0.182  1.305]` — 두 열 모두 -1.2 ~ 1.3 근처로 같은 눈금
- `공식과 같음: True` — (값 - 평균) / 표준편차
- `스케일링 전: 키 몫 0.0% · 용돈 몫 100.0%` — 거리가 사실상 용돈만 봄
- `스케일링 후: 키 몫 62.8% · 용돈 몫 37.2%` — 두 특성이 함께 거리를 결정

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">스케일링 없음</td>
<td style="text-align: left;">단위가 큰 특성이 거리 · 가중합을 <strong>지배</strong></td>
<td style="text-align: left;">트리 모델 (기준값만 보므로 영향 없음)</td>
</tr>
<tr>
<td style="text-align: left;"><code>StandardScaler</code></td>
<td style="text-align: left;">열마다 평균 0 · 표준편차 1</td>
<td style="text-align: left;">거리 · 선형 모델의 기본 선택</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `np.std`의 기본값은 모집단 표준편차(ddof=0)로 `StandardScaler`와 같고, `pandas`의 `.std()`는 ddof=1이라 값이 조금 다름

> ✅ **한 줄 정리** · 단위가 다른 특성은 `StandardScaler`로 열마다 평균 0 · 표준편차 1로 맞춰 거리 · 가중합에서 한 특성이 지배하지 않게 한다.

**딥러닝 코드에서는**

- 신경망 입력도 특성 단위가 다르면 학습이 느리고 불안정 → 표 데이터는 StandardScaler, 이미지는 채널별 mean · std (5-1)

### 2-2. `A` 이상치와 스케일러 3종 · `MinMaxScaler` · `RobustScaler` <sub>원본 03 셀 31 ~ 37</sub>

> 🎯 **이상치가 섞여 있으면 Standard · MinMax · Robust 스케일러의 결과는 어떻게 달라지는가?**

💡 **쉽게 말하면**

- 반 평균을 낼 때 **억만장자 한 명**이 끼면 평균(Standard)도 최댓값(MinMax)도 그 사람에게 끌려감.
- 가운데 값(중앙값, Robust)은 거의 안 움직임.

<img src="images/ml2_scalers.png" alt="이상치가 있을 때 스케일러 3종" width="758">

**실행 흐름**

1. x1 = 1 · 2 · 3 · 100
2. 스케일러 3개 변환 표
3. 정상값 1
4. 3 사이 간격 비교

**먼저 예측** · MinMax로 바꾸면 정상값 1 · 2 · 3은 0 ~ 1 범위 중 어디쯤에 모일까? — 답은 코드 실행 결과 아래에

In [7]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler  # 스케일러 2종 추가


# 원본 셀: 마지막 행이 이상치
X_out = np.array([[1., 10.], [2., 12.], [3., 13.], [100., 1000.]])  # 마지막 행 이상치


# 스케일러 3개로 x1 열 변환
scalers = {"Standard": StandardScaler(), "MinMax": MinMaxScaler(),            # 비교할 3종
           "Robust": RobustScaler()}
cols = {}                                                                     # 스케일러별 결과
for name, sc in scalers.items():                                              # 스케일러마다
    cols[name] = sc.fit_transform(X_out)[:, 0]                                # 첫 열만
print(pd.DataFrame(cols, index=["1", "2", "3", "100"]).round(3).to_string())  # 나란히 표


# 정상값 1 · 2 · 3이 변환 후 얼마나 떨어져 있나 (클수록 구분이 살아 있음)
for name, v in cols.items():                               # 스케일러마다
    print(f"{name:<8} 정상 1 → 3 간격 {v[2] - v[0]:.3f}")  # 정상값 구분이 남았나

     Standard  MinMax  Robust
1      -0.601    0.00  -0.059
2      -0.577    0.01  -0.020
3      -0.554    0.02   0.020
100     1.732    1.00   3.824
Standard 정상 1 → 3 간격 0.047
MinMax   정상 1 → 3 간격 0.020
Robust   정상 1 → 3 간격 0.078


**결과** · 예측 정답은 `0.00` ~ `0.02` — 범위를 이상치 100이 독차지해 정상값이 한 점으로 뭉침

- `1      -0.601    0.00  -0.059` — 정상값의 변환 결과 (Standard · MinMax · Robust)
- `100     1.732    1.00   3.824` — Robust는 이상치를 멀리 보내고 정상값 눈금을 지킴
- `MinMax   정상 1 → 3 간격 0.020` — 정상값 구분이 가장 많이 사라짐
- `Robust   정상 1 → 3 간격 0.078` — 정상값 사이 구분이 가장 잘 남음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>StandardScaler</code></td>
<td style="text-align: left;">평균 · 표준편차 기준 → 이상치가 평균을 끌어당김</td>
<td style="text-align: left;">기본 선택 (분포가 무난할 때)</td>
</tr>
<tr>
<td style="text-align: left;"><code>MinMaxScaler</code></td>
<td style="text-align: left;">최솟값 0 · 최댓값 1 → <strong>이상치 하나가 범위를 독차지</strong></td>
<td style="text-align: left;">범위가 정해진 값 (픽셀 0~255 → 0~1)</td>
</tr>
<tr>
<td style="text-align: left;"><code>RobustScaler</code></td>
<td style="text-align: left;">중앙값 · 사분위 범위(IQR) 기준 → 이상치 영향이 가장 작음</td>
<td style="text-align: left;">이상치가 있는 금액 · 운임 · 센서 값</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - Robust도 이상치를 없애 주지는 않음
> - 이상치를 지울지 · 자를지는 데이터를 보고 따로 판단

> ✅ **한 줄 정리** · 기본은 `StandardScaler`, 이상치가 있으면 `RobustScaler`, 범위가 정해진 값(픽셀)은 `MinMaxScaler`를 후보로 두고 CV로 비교한다.

**딥러닝 코드에서는**

- 이미지 픽셀은 범위가 0~255로 정해져 있어 `/ 255`(MinMax와 같음)가 기본, 그다음 채널 mean · std로 표준화

### 2-3. `A` 머신러닝-1 KNN 숙제 회수 · `스케일러 + KNN` <sub>원본 03 셀 27 · 34 · 35</sub>

> 🎯 **머신러닝-1 6장에서 0.784였던 펭귄 KNN은 스케일링 후 어떻게 되는가?**

💡 **쉽게 말하면**

- KNN은 **가까운 이웃 찾기**인데, 몸무게(g)가 부리 길이(mm)보다 100배 커서 몸무게만 보고 이웃을 찾고 있었음.
- 눈금을 맞추면 네 특성을 다 봄.

**실행 흐름**

1. 머신러닝-1과 같은 분할 · 같은 5등분
2. 스케일링 없음 vs Standard + KNN vs MinMax + KNN

**먼저 예측** · Standard와 MinMax 중 어느 쪽이 더 좋을까? — 답은 코드 실행 결과 아래에

In [8]:
from sklearn.pipeline import Pipeline  # 전처리 + 모델 묶음


# 머신러닝-1 6장과 같은 펭귄 · 같은 분할 (test 봉인)
meas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm",    # 측정값 4열
        "body_mass_g"]
peng = sns.load_dataset("penguins").dropna(subset=meas)            # 머신러닝-1과 같은 342마리
Xp, yp = peng[meas].to_numpy(), peng["species"].to_numpy()         # 표 → 배열
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(                     # 머신러닝-1과 같은 분할
    Xp, yp, test_size=0.2, stratify=yp, random_state=7)
cv_p = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # 같은 5등분


# 스케일링 없음 vs 스케일러 + KNN 묶음 (Pipeline = 3장 · 4장에서 이유)
knn5 = KNeighborsClassifier(n_neighbors=5)                  # 이웃 5개 KNN
cands = {"스케일링 없음": knn5,                             # 비교할 3가지
         "Standard + KNN": Pipeline([("scaler", StandardScaler()),
                                     ("knn", knn5)]),
         "MinMax + KNN": Pipeline([("scaler", MinMaxScaler()),
                                   ("knn", knn5)])}
for name, m in cands.items():                               # 후보마다
    s = cross_val_score(m, Xp_tr, yp_tr, cv=cv_p)           # train 안에서만
    print(f"{name:<14} CV {s.mean():.3f} ± {s.std():.3f}")  # 평균 ± 흔들림

스케일링 없음        CV 0.784 ± 0.023
Standard + KNN CV 0.982 ± 0.016


MinMax + KNN   CV 0.982 ± 0.016


**결과** · 예측 정답은 같음 — 둘 다 `0.982` (원본 셀 34의 붓꽃 한 번 분할에서는 MinMax 0.933 vs Standard 0.911로 달랐음)

- `스케일링 없음        CV 0.784 ± 0.023` — 머신러닝-1 6장과 같은 숫자
- `Standard + KNN CV 0.982 ± 0.016` — 약 0.2 상승 → 트리(0.960)보다도 높음
- `MinMax + KNN   CV 0.982 ± 0.016` — 이 데이터에서는 스케일러 종류보다 **스케일링 여부**가 결정적

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">한 번 나눈 test로 스케일러 비교 (원본 셀 34)</td>
<td style="text-align: left;">MinMax 0.933 vs Standard 0.911 → 운이 섞임</td>
<td style="text-align: left;">결론 내리지 않음</td>
</tr>
<tr>
<td style="text-align: left;">같은 5등분 CV로 비교</td>
<td style="text-align: left;">두 스케일러 모두 0.982 → 차이 없음</td>
<td style="text-align: left;">스케일러 고르기 (기본 선택)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 스케일러 비교도 한 번 나눈 test 점수로 결론 내지 않기 (머신러닝-1 2-2)

> ✅ **한 줄 정리** · 거리 기반 KNN은 스케일링이 성능을 좌우하며, 스케일러 종류는 같은 CV로 비교해 고른다.

**딥러닝 코드에서는**

- 같은 이유로 신경망 입력 정규화를 빼먹으면 학습이 크게 느려짐
- 성능이 이상하게 낮으면 입력 단위부터 확인

### 2-4. `B` 행 길이 맞추기 · `Normalizer` <sub>원본 03 Part 3 셀 38 ~ 45</sub>

> 🎯 **정규화(Normalizer)는 스케일링과 무엇이 다르며, 코사인 유사도와는 어떤 관계인가?**

💡 **쉽게 말하면**

- 록 100곡 · 재즈 50곡을 듣는 A와 10곡 · 5곡을 듣는 B는 **양은 달라도 취향(비율)은 같음**.
- 정규화는 각 행의 길이를 1로 맞춰 방향(취향)만 남김.

**실행 흐름**

1. 벡터 3개 정규화
2. 행 길이 1 확인
3. 두 벡터의 내적 · 정규화 후 내적 · 코사인
4. 문장 2개 코사인

**먼저 예측** · 길이를 1로 맞춘 두 벡터의 내적과 코사인 유사도는 같을까? — 답은 코드 실행 결과 아래에

In [9]:
from sklearn.preprocessing import Normalizer            # 행마다 길이 1
from sklearn.metrics.pairwise import cosine_similarity  # 방향 유사도


# 원본 셀: 행마다 길이를 1로 (스케일링은 열마다, 정규화는 행마다)
X_vec = np.array([[3., 4.], [1., 2.], [10., 0.]])           # 원본 벡터 3개
X_unit = Normalizer().fit_transform(X_vec)                  # 각 행 ÷ 그 행의 길이
print("정규화 후:", X_unit.round(3).tolist())               # [3, 4] → [0.6, 0.8]
print("행 길이:", np.linalg.norm(X_unit, axis=1).round(3))  # 모두 1


# 길이를 1로 맞춘 두 벡터의 내적 = 코사인 유사도 (방향만 비교)
A, B = np.array([[3., 4.]]), np.array([[8., 6.]])  # 방향 · 길이가 다른 두 벡터
A_n, B_n = Normalizer().fit_transform(A), Normalizer().fit_transform(B)  # 각각 길이 1로
print(f"원래 내적 {float(A[0] @ B[0]):.4f}"        # 세 값 비교
      f" | 정규화 후 내적 {float(A_n[0] @ B_n[0]):.4f}"
      f" | 코사인 {float(cosine_similarity(A, B)[0, 0]):.4f}")


# 문장 2개의 단어 개수 벡터 → 코사인 유사도
vec2 = CountVectorizer()                                           # 새 단어장
X_two = vec2.fit_transform(["영화가 좋다", "영화가 진짜로 좋다"])  # 문장 2개 → 개수
print("사전:", vec2.get_feature_names_out().tolist(),              # 사전 · 코사인
      "| 코사인:", round(float(cosine_similarity(X_two[0], X_two[1])[0, 0]), 4))

정규화 후: [[0.6, 0.8], [0.447, 0.894], [1.0, 0.0]]
행 길이: [1. 1. 1.]
원래 내적 48.0000 | 정규화 후 내적 0.9600 | 코사인 0.9600
사전: ['영화가', '좋다', '진짜로'] | 코사인: 0.8165


**결과** · 예측 정답은 같음 — 둘 다 `0.9600`

- `정규화 후: [[0.6, 0.8], [0.447, 0.894], [1.0, 0.0]]` — [3, 4] → 길이 5로 나눠 [0.6, 0.8]
- `행 길이: [1. 1. 1.]` — 행마다 길이 1
- `원래 내적 48.0000 | 정규화 후 내적 0.9600 | 코사인 0.9600` — 길이가 1이면 내적 = 코사인
- `사전: ['영화가', '좋다', '진짜로'] | 코사인: 0.8165` — 단어가 2개 겹침 → 방향이 비슷 (의미가 같다는 뜻은 아님)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">스케일링 (Standard · MinMax · Robust)</td>
<td style="text-align: left;"><strong>열</strong>(특성)마다 단위 맞추기</td>
<td style="text-align: left;">특성끼리 단위가 다를 때</td>
</tr>
<tr>
<td style="text-align: left;">정규화 (<code>Normalizer</code>)</td>
<td style="text-align: left;"><strong>행</strong>(샘플)마다 길이 1 → 방향만 남김</td>
<td style="text-align: left;">문장 · 임베딩 비교, 코사인 유사도</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 스케일링은 **열**, 정규화는 **행**
> - 이름이 비슷해 가장 많이 헷갈림

> ✅ **한 줄 정리** · 정규화는 행마다 길이를 1로 맞춰 방향만 비교하게 하며, 그때 내적이 곧 코사인 유사도다.

**딥러닝 코드에서는**

- 문장 임베딩 검색 · 추천에서 `F.normalize(emb, dim=1)` 후 `emb @ emb.T` = 코사인 유사도 행렬

### 2-더 알아보기 · 인코더 · 스케일러 선택표

1장(인코딩) · 2장(스케일링)의 비교를 **데이터 상황 → 고를 것** 하나의 표로 모음.
섹션 비교표는 "무엇이 다른가", 이 표는 "내 데이터에는 무엇을 쓰나".

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">데이터 상황</th>
<th style="text-align: left;">고를 것</th>
<th style="text-align: left;">이유</th>
<th style="text-align: left;">자세히</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">순서 없는 범주 X (색 · 항구 · 성별)</td>
<td style="text-align: left;"><code>OneHotEncoder(handle_unknown="ignore")</code></td>
<td style="text-align: left;">가짜 순서 없음 · 범주를 기억</td>
<td style="text-align: left;">1-1 · 1-2</td>
</tr>
<tr>
<td style="text-align: left;">진짜 순서가 있는 범주 X (크기 · 등급)</td>
<td style="text-align: left;"><code>OrdinalEncoder(categories=[[...]])</code></td>
<td style="text-align: left;">순서를 직접 지정</td>
<td style="text-align: left;">1-3</td>
</tr>
<tr>
<td style="text-align: left;">정답 y (품종 · 클래스 이름)</td>
<td style="text-align: left;"><code>LabelEncoder</code></td>
<td style="text-align: left;">번호 ↔ 이름 왕복</td>
<td style="text-align: left;">1-3</td>
</tr>
<tr>
<td style="text-align: left;">한 번 보고 끝나는 탐색 · 그래프</td>
<td style="text-align: left;"><code>pd.get_dummies</code></td>
<td style="text-align: left;">짧지만 기준을 기억 못 함</td>
<td style="text-align: left;">1-1</td>
</tr>
<tr>
<td style="text-align: left;">범주가 아주 많음 (단어 · ID) · 딥러닝</td>
<td style="text-align: left;">번호 + <code>nn.Embedding</code></td>
<td style="text-align: left;">원-핫 열 폭발 방지</td>
<td style="text-align: left;">5-2</td>
</tr>
<tr>
<td style="text-align: left;">문장</td>
<td style="text-align: left;"><code>CountVectorizer</code> (train 사전)</td>
<td style="text-align: left;">새 문장은 transform만</td>
<td style="text-align: left;">1-4</td>
</tr>
<tr>
<td style="text-align: left;">숫자 X · 분포 무난 · 거리 · 선형 모델</td>
<td style="text-align: left;"><code>StandardScaler</code></td>
<td style="text-align: left;">기본 선택</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">숫자 X에 이상치 (운임 · 금액)</td>
<td style="text-align: left;"><code>RobustScaler</code></td>
<td style="text-align: left;">중앙값 · IQR 기준</td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">범위가 정해진 값 (픽셀 0 ~ 255)</td>
<td style="text-align: left;"><code>MinMaxScaler</code> (= <code>/ 255</code>)</td>
<td style="text-align: left;">0 ~ 1로</td>
<td style="text-align: left;">2-2 · 5-1</td>
</tr>
<tr>
<td style="text-align: left;">트리 모델</td>
<td style="text-align: left;">스케일링 생략</td>
<td style="text-align: left;">기준값만 봄</td>
<td style="text-align: left;">6</td>
</tr>
<tr>
<td style="text-align: left;">방향만 비교 (문장 · 임베딩)</td>
<td style="text-align: left;"><code>Normalizer</code></td>
<td style="text-align: left;">행 길이 1 → 코사인</td>
<td style="text-align: left;">2-4</td>
</tr>
</tbody>
</table>

---
## 3. fit · transform · 누수

> **이 장의 흐름**
>
> - 1 · 2장의 인코더 · 스케일러는 모두 데이터를 보고 기준을 만든다(fit)
> - 그 기준을 어느 데이터로 만드느냐에 따라 평가가 공정한지가 갈림.
> - 머신러닝-1의 'test는 마지막 1회'를 전처리까지 넓힘

**파이프라인 위치** · ③ 전처리 × ⑤ 교차검증 — 기준은 train으로만

### 3-1. `A` 기준 만들기와 적용하기 · `fit` · `transform` · `fit_transform` <sub>원본 03 셀 46 ~ 48</sub>

> 🎯 **`fit` · `transform` · `fit_transform`은 각각 무엇을 하며, train과 test에 어떻게 나눠 쓰는가?**

💡 **쉽게 말하면**

- `fit`은 **자 만들기**(눈금 계산), `transform`은 **그 자로 재기**.
- 자는 교과서(train)로 한 번만 만들고, 시험지(test)도 같은 자로 잼.

**실행 흐름**

1. 붓꽃 7 : 3
2. train으로 fit
3. 둘 다 transform
4. 변환 후 평균
5. fit_transform 비교
6. test 처리 3가지로 KNN 채점

**먼저 예측** · train 기준으로 바꾼 test의 평균은 0일까? — 답은 코드 실행 결과 아래에

In [10]:
from sklearn.datasets import load_iris  # 붓꽃 데이터


# 원본 Part 4와 같은 붓꽃 7 : 3 분할
X_iris, y_iris = load_iris(return_X_y=True)  # 문제 · 정답만
X_tr, X_te, y_tr, y_te = train_test_split(   # 원본과 같은 7 : 3
    X_iris, y_iris, test_size=0.3, stratify=y_iris, random_state=42)


# 규칙 1: fit은 train에만 → 기준(평균 · 표준편차)은 train에서
sc = StandardScaler().fit(X_tr)               # 자는 train으로만
print("train 기준 평균:", sc.mean_.round(3))  # 자의 눈금


# 규칙 2: transform은 train · test 둘 다 같은 기준으로
X_tr_ok, X_te_ok = sc.transform(X_tr), sc.transform(X_te)      # 같은 자로 둘 다
print("변환 후 평균 → train:", X_tr_ok.mean(axis=0).round(2),  # test는 0이 아니어도 정상
      "| test:", X_te_ok.mean(axis=0).round(2))                # test가 0이 아닌 게 정상


# fit_transform = fit + transform을 한 번에 (train에서만 씀)
print("fit_transform과 같음:",  # 한 번에 해도 같은 값
      np.allclose(StandardScaler().fit_transform(X_tr), X_tr_ok))


# 같은 KNN으로 세 가지 test 처리 비교
knn = KNeighborsClassifier(n_neighbors=5).fit(X_tr_ok, y_tr)      # 변환된 train으로 학습
X_te_own = StandardScaler().fit_transform(X_te)                   # 실수 1: test 자기 기준
print(f"train 기준 transform : {knn.score(X_te_ok, y_te):.3f}")   # 올바른 절차
print(f"test에 따로 fit      : {knn.score(X_te_own, y_te):.3f}")  # 다른 자 (실수 1)
print(f"transform 빠뜨림     : {knn.score(X_te, y_te):.3f}")      # 실수 2

train 기준 평균: [5.873 3.055 3.785 1.206]
변환 후 평균 → train: [ 0. -0. -0. -0.] | test: [-0.12  0.02 -0.05 -0.03]
fit_transform과 같음: True
train 기준 transform : 0.911
test에 따로 fit      : 0.956
transform 빠뜨림     : 0.333


**결과** · 예측 정답은 아님 — `[-0.12 0.02 -0.05 -0.03]`처럼 0 근처일 뿐 (0이 아닌 게 정상)

- `train 기준 평균: [5.873 3.055 3.785 1.206]` — fit이 계산한 자의 눈금
- `변환 후 평균 → train: [ 0. -0. -0. -0.] | test: [-0.12  0.02 -0.05 -0.03]` — train은 정확히 0, test는 다른 꽃들이라 조금 다름
- `fit_transform과 같음: True` — fit + transform을 한 번에
- `train 기준 transform : 0.911` — 올바른 절차의 점수
- `test에 따로 fit      : 0.956` — 점수는 더 높지만 test 45송이 전체 통계를 미리 본 것 (실제 서비스에선 한 명씩 들어와 불가능)
- `transform 빠뜨림     : 0.333` — 단위가 다른 숫자를 받아 찍기 수준 → 에러 없이 조용히 망가짐

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>fit(X_train)</code></td>
<td style="text-align: left;">기준(평균 · 표준편차 · 범주 · 사전) <strong>계산만</strong></td>
<td style="text-align: left;">train에 한 번</td>
</tr>
<tr>
<td style="text-align: left;"><code>transform(X)</code></td>
<td style="text-align: left;">저장된 기준으로 <strong>변환만</strong></td>
<td style="text-align: left;">train · valid · test · 실제 입력 모두</td>
</tr>
<tr>
<td style="text-align: left;"><code>fit_transform(X_train)</code></td>
<td style="text-align: left;">fit + transform을 한 번에 (결과 같음)</td>
<td style="text-align: left;">train에서만 (test에 쓰면 실수 1)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - "test는 fit하지 않는다" ≠ "test는 건드리지 않는다"
> - `transform`은 반드시 적용

> ✅ **한 줄 정리** · `fit`은 train에만 해서 기준을 만들고, `transform`은 train · valid · test 모두에 같은 기준으로 적용한다.

**딥러닝 코드에서는**

- 학습 후 `scaler`를 모델과 함께 저장(`joblib.dump`)해야 배포에서 같은 자로 잼
- 이미지라면 mean · std 상수를 저장 (5-1)

### 3-2. `A` 누수 있는 전처리 vs Pipeline · `Pipeline` <sub>원본 03 Part 4 셀 49 ~ 60</sub>

> 🎯 **전체 데이터로 스케일러를 먼저 맞춘 뒤 교차검증하면 무엇이 잘못되며, `Pipeline`은 어떻게 막는가?**

💡 **쉽게 말하면**

- 모의고사 문제까지 보고 만든 자로 채점하는 것.
- 점수가 **올라가든 안 올라가든** 시험지를 미리 본 순간 그 채점은 무효.

<img src="images/ml2_leak.png" alt="누수: 자를 언제 만드는가" width="837">

**실행 흐름**

1. (누수) 붓꽃 150개로 scaler fit
2. CV / (올바름) Pipeline CV
3. KNN · LR 비교
4. fold별 scaler 평균 확인

**먼저 예측** · 누수 방식의 KNN 점수는 Pipeline보다 높을까? — 답은 코드 실행 결과 아래에

In [11]:
from sklearn.model_selection import cross_validate  # fold별 모델도 돌려받기


# 원본 셀: (누수) 붓꽃 전체 150개로 scaler를 먼저 fit → 그다음 교차검증
X_iris_leak = StandardScaler().fit_transform(X_iris)  # 검증 fold 정보까지 섞임


# (올바름) Pipeline → fold마다 학습 4등분으로만 scaler fit
pipes = {"KNN": KNeighborsClassifier(n_neighbors=5),                   # 비교할 모델 2개
         "LR": LogisticRegression(max_iter=5000, random_state=42)}
for name, model in pipes.items():                                      # 모델마다
    leak = cross_val_score(model, X_iris_leak, y_iris, cv=cv5)         # 미리 변환한 X로
    pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])  # 전처리를 안에 넣음
    ok = cross_val_score(pipe, X_iris, y_iris, cv=cv5)                 # 원본 X로 · fold마다 fit
    print(f"{name:<3} 누수 {leak.mean():.3f} ± {leak.std():.3f}"       # 두 방식 비교
          f" | Pipeline {ok.mean():.3f} ± {ok.std():.3f}")


# Pipeline이 fold마다 scaler를 새로 fit했는지 확인 (fold별 평균이 다름)
res = cross_validate(Pipeline([("scaler", StandardScaler()),     # 학습된 fold 모델 보관
                               ("model", pipes["KNN"])]),
                     X_iris, y_iris, cv=cv5, return_estimator=True)
fold_means = [est.named_steps["scaler"].mean_[0] for est in res["estimator"]]  # fold별 자
print("fold별 꽃받침 길이 평균:", np.round(fold_means, 3))       # 5개가 다름
print("전체 150개 평균       :", round(X_iris[:, 0].mean(), 3))  # 누수 방식의 자

KNN 누수 0.967 ± 0.037 | Pipeline 0.973 ± 0.025


LR  누수 0.953 ± 0.045 | Pipeline 0.953 ± 0.045


fold별 꽃받침 길이 평균: [5.814 5.879 5.852 5.833 5.839]
전체 150개 평균       : 5.843


**결과** · 예측 정답은 아님 — 누수 `0.967` < Pipeline `0.973`, 로지스틱은 `0.953`으로 같음

- `KNN 누수 0.967 ± 0.037 | Pipeline 0.973 ± 0.025` — 누수가 점수를 올린다는 보장도 없음
- `LR  누수 0.953 ± 0.045 | Pipeline 0.953 ± 0.045` — 점수가 같아도 절차는 틀림
- `fold별 꽃받침 길이 평균: [5.814 5.879 5.852 5.833 5.839]` — Pipeline은 fold마다 학습 4등분으로 새 자를 만듦
- `전체 150개 평균       : 5.843` — 누수 방식은 5개 fold 모두 이 하나의 자 (검증 fold 포함)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">누수 있는 전처리</td>
<td style="text-align: left;">전체 X로 fit → 검증 fold 정보가 기준에 섞임</td>
<td style="text-align: left;">쓰지 않음 (점수가 같아 보여도)</td>
</tr>
<tr>
<td style="text-align: left;"><code>Pipeline</code> + <code>cross_val_score</code></td>
<td style="text-align: left;">fold마다 학습 부분으로만 fit → 검증 fold는 transform만</td>
<td style="text-align: left;">교차검증 · GridSearchCV (항상)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 누수 여부는 **점수 차이가 아니라 절차**로 판단
> - 점수가 비슷하다고 괜찮은 것이 아님

> ✅ **한 줄 정리** · 전처리를 `Pipeline`에 넣고 Pipeline째 교차검증하면 fold마다 학습 부분으로만 fit되어 누수가 막힌다.

**딥러닝 코드에서는**

- 딥러닝에서도 정규화 통계 · 단어 사전 · 결측 채움 값을 **train 폴더만으로** 계산해 val · test에 적용

### 3-3. `A` 없는 패턴을 만드는 누수 · `SelectKBest` <sub>원본 03 셀 61 ~ 63</sub>

> 🎯 **전처리 단계에서 정답을 보고 특성을 고르면 어떤 일이 생기는가?**

💡 **쉽게 말하면**

- 시험 문제를 먼저 보고 **나올 단원만 골라 외운** 뒤 '실력이 87점'이라고 하는 것.
- 정답과 아무 관계 없는 난수에서도 높은 점수가 나옴.

**실행 흐름**

1. 난수 X (100 × 5000) · 동전 y
2. (누수) 전체로 열 20개 선택 후 CV
3. (올바름) Pipeline 안에서 선택
4. 정답 0.5와 비교

**먼저 예측** · X와 y가 아무 관계 없을 때 누수 방식의 CV 점수는 0.5 근처일까? — 답은 코드 실행 결과 아래에

In [12]:
from sklearn.feature_selection import SelectKBest, f_classif  # 정답과 관련 높은 열 고르기


# 원본 셀: 정답과 아무 관계 없는 난수 X (100행 × 5000열) · 동전 던지기 y
rng = np.random.RandomState(42)        # 원본과 같은 난수
X_rand = rng.normal(size=(100, 5000))  # 아무 의미 없는 5000열
y_rand = np.array([0, 1] * 50)         # 정확히 50 : 50
rng.shuffle(y_rand)                    # 정답 섞기


# (누수) 전체 100행을 보고 y와 관련 있어 보이는 열 20개를 먼저 고름
X_sel = SelectKBest(f_classif, k=20).fit_transform(X_rand, y_rand)  # 전체 정답을 보고 고름
lr1000 = LogisticRegression(max_iter=1000)                          # 같은 모델
leak = cross_val_score(lr1000, X_sel, y_rand, cv=cv5)               # 이미 고른 열로 CV


# (올바름) Pipeline 안에서 fold마다 학습 4등분으로만 고름
pipe_sel = Pipeline([("select", SelectKBest(f_classif, k=20)),  # 고르기도 안으로
                     ("lr", lr1000)])
ok = cross_val_score(pipe_sel, X_rand, y_rand, cv=cv5)          # fold마다 다시 고름
print(f"누수     : {leak.mean():.3f} ± {leak.std():.3f}")       # 없는 패턴
print(f"Pipeline : {ok.mean():.3f} ± {ok.std():.3f}")           # 정직한 점수
print("정답     : 0.500 (X와 y는 아무 관계 없음)")              # 찍기 수준

누수     : 0.870 ± 0.081
Pipeline : 0.520 ± 0.060
정답     : 0.500 (X와 y는 아무 관계 없음)


**결과** · 예측 정답은 아님 — `0.870`, 존재하지 않는 패턴이 87%로 보임

- `누수     : 0.870 ± 0.081` — 수천 개 열 중 우연히 y와 맞아 보이는 20열을 전체 정답으로 골랐기 때문
- `Pipeline : 0.520 ± 0.060` — fold마다 학습 부분으로만 골라 정직한 찍기 수준
- `정답     : 0.500 (X와 y는 아무 관계 없음)` — 진짜 실력

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">변환기 누수 (스케일 · 채우기)</td>
<td style="text-align: left;">점수 차이가 <strong>작을 수도</strong> 있음 (3-2)</td>
<td style="text-align: left;">그래도 절차 위반</td>
</tr>
<tr>
<td style="text-align: left;">선택 누수 (특성 고르기 · 임곗값 고르기)</td>
<td style="text-align: left;">정답을 보고 고르므로 <strong>없는 성능을 만듦</strong> (0.870)</td>
<td style="text-align: left;">선택도 반드시 Pipeline 안 또는 val로</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 특성 선택 · 이상치 제거 기준 · 임곗값 고르기처럼 **정답 y를 보는** 결정은 반드시 train 안에서 (임곗값은 머신러닝-3)

> ✅ **한 줄 정리** · 정답을 보고 고르는 전처리를 Pipeline 밖에서 하면 없는 성능이 만들어지므로, 선택 단계도 Pipeline 안에 넣는다.

**딥러닝 코드에서는**

- 특성이 많고 샘플이 적은 의료 · 유전자 데이터에서 실제로 자주 생기는 사고
- 딥러닝에서는 val로 고른 것을 test로 다시 확인 (머신러닝-1 5-3)

---
## 4. 파이프라인

> **이 장의 흐름**
>
> - 3장에서 기준을 train으로만 만들어야 한다는 규칙을 봤다
> - 결측 · 인코딩 · 스케일링이 열마다 섞이면 사람이 매번 지키기 어려움.
> - 실제 표(타이타닉)로 전부를 하나의 Pipeline에 묶음

<img src="images/ml2_pipeline.png" alt="Pipeline 구조" width="912">

**파이프라인 위치** · ① 데이터 준비 → ③ 전처리 → ④ 모델 → ⑤ 탐색을 한 객체로

### 4-1. `A` Pipeline 구조와 설정 이름 · `named_steps` · `이름__설정` <sub>원본 03 셀 52 · 미션 3 정답1 셀 16 ~ 17</sub>

> 🎯 **`Pipeline` 안의 단계는 어떻게 꺼내고, 안의 설정은 GridSearchCV에 어떤 이름으로 넘기는가?**

💡 **쉽게 말하면**

- Pipeline은 **이름표 붙은 조립 라인**.
- `fit` 버튼 한 번에 앞 칸부터 차례로 돌고, 칸 이름으로 안을 들여다보거나 설정을 바꿈.

**실행 흐름**

1. 펭귄 scaler + KNN 조립
2. fit
3. named_steps로 scaler 확인
4. knn__n_neighbors 후보 6개를 GridSearchCV로

**먼저 예측** · Pipeline 안 KNN의 이웃 수를 탐색할 때 설정 이름은? — 답은 코드 실행 결과 아래에

In [13]:
from sklearn.model_selection import GridSearchCV  # 설정 탐색기


# Pipeline = [(이름, 단계), ...] → 마지막 단계만 모델, 앞은 전부 변환기
pipe_knn = Pipeline([("scaler", StandardScaler()),  # 이름표 붙은 단계 2개
                     ("knn", KNeighborsClassifier(n_neighbors=5))])
pipe_knn.fit(Xp_tr, yp_tr)                          # scaler.fit_transform → knn.fit 순서로 자동


# 안의 단계 꺼내기: named_steps["이름"] → train 273마리 기준으로 fit됨
inner = pipe_knn.named_steps["scaler"]           # 이름으로 단계 꺼내기
print("scaler 평균 = train 평균:", np.allclose(inner.mean_, Xp_tr.mean(axis=0)))  # train으로 fit?
print("단계 이름:", list(pipe_knn.named_steps))  # 붙인 이름 목록


# 설정 탐색: '단계이름__설정' 형식 (밑줄 2개)
ks = [1, 5, 15, 30, 60, 100]                                            # 이웃 수 후보
grid = GridSearchCV(pipe_knn, {"knn__n_neighbors": ks},                 # 단계이름__설정
                    cv=cv_p).fit(Xp_tr, yp_tr)
means = grid.cv_results_["mean_test_score"].round(3).tolist()           # 후보별 CV
print("k별 CV:", dict(zip(ks, means)))                                  # k가 크면 하락
print("최적:", grid.best_params_, "| CV:", round(grid.best_score_, 3))  # 고른 k

scaler 평균 = train 평균: True
단계 이름: ['scaler', 'knn']


k별 CV: {1: 0.982, 5: 0.982, 15: 0.982, 30: 0.978, 60: 0.953, 100: 0.828}
최적: {'knn__n_neighbors': 15} | CV: 0.982


**결과** · 예측 정답은 `knn__n_neighbors` — 단계 이름 + 밑줄 2개 + 설정 이름

- `scaler 평균 = train 평균: True` — 안의 scaler는 train 펭귄으로만 fit됨
- `단계 이름: ['scaler', 'knn']` — 내가 붙인 이름 그대로
- `k별 CV: {1: 0.982, 5: 0.982, 15: 0.982, 30: 0.978, 60: 0.953, 100: 0.828}` — k가 너무 크면 먼 이웃까지 투표 → 과소적합 (머신러닝-1 4-1)
- `최적: {'knn__n_neighbors': 15} | CV: 0.982` — 탐색 중에도 fold마다 scaler가 새로 fit됨

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">단계를 따로 실행</td>
<td style="text-align: left;"><code>fit</code> · <code>transform</code>을 순서대로 직접 → 빠뜨리기 쉬움</td>
<td style="text-align: left;">원리를 볼 때만</td>
</tr>
<tr>
<td style="text-align: left;"><code>Pipeline([...])</code></td>
<td style="text-align: left;"><code>fit</code> 한 번 = 앞 단계 fit_transform → 마지막 모델 fit</td>
<td style="text-align: left;">전처리가 있는 모든 모델 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>make_pipeline(...)</code></td>
<td style="text-align: left;">이름을 자동으로 (<code>standardscaler</code> 등)</td>
<td style="text-align: left;">이름을 직접 부를 일이 없을 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 밑줄은 **2개** (`knn_n_neighbors`는 에러), 마지막 단계만 모델이고 앞 단계는 모두 `transform`이 있어야 함

> ✅ **한 줄 정리** · `Pipeline([(이름, 단계), ...])`로 전처리와 모델을 묶고, 안의 설정은 `이름__설정`으로 탐색한다.

**딥러닝 코드에서는**

- PyTorch의 `nn.Sequential(...)` · `transforms.Compose([...])`가 같은 '차례로 통과' 구조
- `model[0]`처럼 단계를 꺼내 봄

### 4-2. `A` 빈칸 채우기 · `SimpleImputer` <sub>원본 4-1 로지스틱 Part 2 · 미션 3 정답1 셀 9 ~ 10</sub>

> 🎯 **결측값은 어떤 값으로 채우며, 그 값은 어느 데이터로 계산해야 하는가?**

💡 **쉽게 말하면**

- 빈칸 채우기도 **자 만들기**.
- '빈 나이는 중앙값으로'의 중앙값을 train에서 계산해 기억해 두고, valid · test의 빈칸도 그 값으로 채움.

**실행 흐름**

1. Cabin1 만들기 · 열 정리
2. 3분할 (test 봉인)
3. train 결측 수
4. SimpleImputer fit (median · most_frequent)
5. 전체 중앙값과 비교
6. fillna와 비교

**먼저 예측** · train 534명으로 계산한 나이 중앙값은 전체 891명의 중앙값과 같을까? — 답은 코드 실행 결과 아래에

In [14]:
from sklearn.impute import SimpleImputer  # 빈칸 채우기 도구


# 4-1 로지스틱 Part 2: 객실 첫 글자(Cabin1) 만들고 쓰지 않을 열 제거
X_t = titanic.drop(columns="Survived").copy()                       # 정답 열 빼기
X_t["Cabin1"] = X_t["Cabin"].fillna("N").str[:1]                    # 결측 → N, 첫 글자만
X_t = X_t.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])  # 안 쓸 열 제거
y_t = titanic["Survived"]                                           # 정답


# 3분할 (4-1 Part 2와 같음): test 20% 봉인 → 남은 것을 train 75% · valid 25%
X_dev, X_test, y_dev, y_test = train_test_split(                            # test 20% 봉인
    X_t, y_t, test_size=0.2, stratify=y_t, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(  # 나머지를 train · valid로
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=42)
print(f"train {len(X_train)} | valid {len(X_valid)} | test {len(X_test)}")  # 534 · 178 · 179
print("train 결측:", X_train[["Age", "Embarked"]].isna().sum().to_dict())   # train 빈칸 수


# SimpleImputer: fit → train으로 채울 값 계산, transform → 빈칸 채우기
age_imp = SimpleImputer(strategy="median").fit(X_train[["Age"]])              # train 중앙값 기억
emb_imp = SimpleImputer(strategy="most_frequent").fit(X_train[["Embarked"]])  # train 최빈값
print("채울 값 → Age:", age_imp.statistics_[0],                               # 기억한 값
      "| Embarked:", emb_imp.statistics_[0])
print("비교 → 전체 891명 Age 중앙값:", titanic["Age"].median())               # 누수 기준


# fillna와 같은 결과 (값을 직접 들고 다녀야 하는 차이)
filled = age_imp.transform(X_valid[["Age"]]).ravel()                        # valid 빈칸 채우기
same = np.allclose(filled, X_valid["Age"].fillna(X_train["Age"].median()))  # fillna와 비교
print("valid 결측", X_valid["Age"].isna().sum(), "개 채움 · fillna와 같음:", same)  # 결과

train 534 | valid 178 | test 179
train 결측: {'Age': 98, 'Embarked': 2}
채울 값 → Age: 29.0 | Embarked: S
비교 → 전체 891명 Age 중앙값: 28.0
valid 결측 39 개 채움 · fillna와 같음: True


**결과** · 예측 정답은 다름 — train `29.0` vs 전체 `28.0` (전체로 계산하면 test 정보가 섞임)

- `train 534 | valid 178 | test 179` — 4-1 로지스틱 Part 2와 같은 3분할
- `train 결측: {'Age': 98, 'Embarked': 2}` — train 안에서만 센 빈칸
- `채울 값 → Age: 29.0 | Embarked: S` — `statistics_`에 기억된 값
- `비교 → 전체 891명 Age 중앙값: 28.0` — test까지 섞은 값 = 누수 기준
- `valid 결측 39 개 채움 · fillna와 같음: True` — 결과는 같지만 Imputer는 값을 스스로 들고 다님

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>df.fillna(값)</code></td>
<td style="text-align: left;">채울 값을 <strong>직접 계산 · 보관</strong>해야 함</td>
<td style="text-align: left;">탐색 · 한 번 쓰고 끝나는 정리</td>
</tr>
<tr>
<td style="text-align: left;"><code>SimpleImputer</code></td>
<td style="text-align: left;">fit에서 값을 계산해 <code>statistics_</code>에 <strong>기억</strong>, Pipeline 안에 들어감</td>
<td style="text-align: left;">모델 입력 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>strategy="median"</code> / <code>"mean"</code></td>
<td style="text-align: left;">중앙값은 이상치에 강함 / 평균</td>
<td style="text-align: left;">숫자 열 (운임처럼 치우치면 median)</td>
</tr>
<tr>
<td style="text-align: left;"><code>strategy="most_frequent"</code></td>
<td style="text-align: left;">가장 많이 나온 값</td>
<td style="text-align: left;">글자 열 (Embarked)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `Cabin`은 77%가 비어 있어 값으로 채우기보다 '비어 있음' 자체를 범주 `N`으로 (Cabin1)

> ✅ **한 줄 정리** · 결측은 `SimpleImputer`로 train에서 계산한 중앙값 · 최빈값으로 채우고, 그 값을 valid · test에도 그대로 쓴다.

**딥러닝 코드에서는**

- 시계열 · 센서 데이터의 빈칸도 train 구간 통계로만 채움
- 뒤쪽 미래 값으로 앞을 채우면(`bfill`) 누수

### 4-3. `A` 열마다 다른 전처리 · `ColumnTransformer` <sub>원본 4-1 로지스틱 Part 2 셀 8</sub>

> 🎯 **숫자 열과 글자 열이 섞인 표에 열마다 다른 전처리를 한 번에 적용하려면?**

💡 **쉽게 말하면**

- `ColumnTransformer`는 **분류 작업대**.
- 숫자 열은 왼쪽 라인(채우기 → 표준화), 글자 열은 오른쪽 라인(채우기 → 원-핫)으로 보낸 뒤 결과를 옆으로 이어 붙임.

**실행 흐름**

1. 숫자 5열 · 글자 3열 목록
2. 열 종류별 Pipeline 2개
3. ColumnTransformer
4. train fit
5. valid transform
6. 열 이름 · 빈칸 확인

**먼저 예측** · 8개 열이 전처리 후 몇 개 열이 될까? — 답은 코드 실행 결과 아래에

In [15]:
from sklearn.compose import ColumnTransformer  # 열마다 다른 전처리


# 열 종류별 목록 (4-1 Part 2와 같음)
num_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare"]  # 숫자 5열
cat_cols = ["Sex", "Embarked", "Cabin1"]                # 글자 3열


# 숫자 열: 중앙값 채우기 → 표준화 / 글자 열: 최빈값 채우기 → 원-핫
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),         # 숫자 라인
                     ("scaler", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),  # 글자 라인
                     ("onehot", OneHotEncoder(handle_unknown="ignore"))])
prep = ColumnTransformer([("num", num_pipe, num_cols),                      # 두 라인을 한 묶음으로
                          ("cat", cat_pipe, cat_cols)],
                         remainder="drop")                                  # 목록에 없는 열은 버림


# train으로 fit → train · valid 변환 (열 개수가 같아야 함)
Z_train = prep.fit_transform(X_train)                     # train으로 fit + 변환
Z_valid = prep.transform(X_valid)                         # valid는 변환만
names = prep.get_feature_names_out()                      # 결과 열 이름
print("변환 전:", X_train.shape, "→ 후:", Z_train.shape, "| valid:", Z_valid.shape)  # 8 → 19열
print("앞 6개 열:", names[:6].tolist())                   # 라인이름__열이름
print("Cabin1 열:", [n for n in names if "Cabin1" in n])  # 객실 첫 글자 9종
print("빈칸 남음:", np.isnan(Z_train).sum())              # 0이 적어 일반 배열로 나옴

변환 전: (534, 8) → 후: (534, 19) | valid: (178, 19)
앞 6개 열: ['num__Pclass', 'num__Age', 'num__SibSp', 'num__Parch', 'num__Fare', 'cat__Sex_female']
Cabin1 열: ['cat__Cabin1_A', 'cat__Cabin1_B', 'cat__Cabin1_C', 'cat__Cabin1_D', 'cat__Cabin1_E', 'cat__Cabin1_F', 'cat__Cabin1_G', 'cat__Cabin1_N', 'cat__Cabin1_T']
빈칸 남음: 0


**결과** · 예측 정답은 `19`개 — 숫자 5 + Sex 2 + Embarked 3 + Cabin1 9

- `변환 전: (534, 8) → 후: (534, 19) | valid: (178, 19)` — valid도 train과 같은 19열
- `앞 6개 열: ['num__Pclass', 'num__Age', 'num__SibSp', 'num__Parch', 'num__Fare', 'cat__Sex_female']` — `라인이름__열이름`으로 출처가 남음
- `Cabin1 열: ['cat__Cabin1_A', 'cat__Cabin1_B', 'cat__Cabin1_C', 'cat__Cabin1_D', 'cat__Cabin1_E', 'cat__Cabin1_F', 'cat__Cabin1_G', 'cat__Cabin1_N', 'cat__Cabin1_T']` — train에 나온 객실 첫 글자 9종
- `빈칸 남음: 0` — 숫자 · 글자 빈칸 모두 채워짐

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">열마다 따로 변환 후 <code>np.hstack</code></td>
<td style="text-align: left;">열 순서 · 이름을 직접 관리, test에 다시 반복해야 함</td>
<td style="text-align: left;">쓰지 않음</td>
</tr>
<tr>
<td style="text-align: left;"><code>ColumnTransformer</code></td>
<td style="text-align: left;">열 목록마다 다른 Pipeline → 결과를 자동으로 이어 붙임</td>
<td style="text-align: left;">숫자 · 글자 열이 섞인 표 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>remainder="drop"</code> / <code>"passthrough"</code></td>
<td style="text-align: left;">목록에 없는 열 버림 / 그대로 통과</td>
<td style="text-align: left;">필요한 열만 / 이미 숫자인 열을 그대로</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 열 이름 목록에 빠진 열은 `remainder="drop"`으로 조용히 버려짐
> - 결과 열 수를 꼭 확인

> ✅ **한 줄 정리** · `ColumnTransformer`로 열 종류마다 다른 Pipeline을 지정하면 fit 한 번에 표 전체가 모델 입력 배열이 된다.

**딥러닝 코드에서는**

- 표 + 이미지 같은 여러 입력 모델에서도 입력 종류별로 다른 전처리를 한 뒤 이어 붙이는(concat) 구조가 같음

### 4-4. `A` 전처리 설정까지 탐색 · `GridSearchCV + Pipeline` <sub>추가 · 머신러닝-1 4-2 연결</sub>

> 🎯 **전처리와 모델을 묶은 뒤, 모델 설정과 전처리 설정을 함께 교차검증으로 고르려면?**

💡 **쉽게 말하면**

- 조립 라인 전체를 **자동 실험기**에 넣는 것.
- 모델 설정(`C`)뿐 아니라 안쪽 칸의 채우기 방법까지 칸 주소로 지정해 모든 조합을 시험함.

**실행 흐름**

1. ColumnTransformer + 로지스틱
2. 후보 C 4개 × 채우기 2개 = 8개
3. GridSearchCV
4. C별 점수

**먼저 예측** · 규제 세기 `C`가 가장 작은 0.01(규제 가장 강함)이 가장 좋을까? — 답은 코드 실행 결과 아래에

In [16]:
# 전처리 + 모델을 하나로 → fit 한 번에 열마다 다른 전처리 + 학습
lr_pipe = Pipeline([("prep", prep),  # 전처리 + 로지스틱
                    ("model", LogisticRegression(max_iter=5000))])


# 이름__이름__설정: 안쪽 단계까지 경로로 지정 (prep → num → imputer)
space = {"model__C": [0.01, 0.1, 1, 10],                             # 모델 설정 + 전처리 설정
         "prep__num__imputer__strategy": ["median", "mean"]}
grid_t = GridSearchCV(lr_pipe, space, cv=cv5).fit(X_train, y_train)  # 8개 조합 · train 안에서
print("후보:", len(grid_t.cv_results_["params"]), "개 → 최적:",      # 고른 조합
      grid_t.best_params_)
print(f"최적 CV: {grid_t.best_score_:.3f}")                          # 그 점수


# C별 결과 (채우기는 최적 방법으로 고정해 비교)
res_t = pd.DataFrame(grid_t.cv_results_)                        # 전체 결과 표
best_imp = grid_t.best_params_["prep__num__imputer__strategy"]  # 고른 채우기 방법
pick = res_t["param_prep__num__imputer__strategy"] == best_imp  # 그 방법의 행만
print(res_t.loc[pick, ["param_model__C", "mean_test_score"]]    # C별 점수
      .round(3).to_string(index=False))

후보: 8 개 → 최적: {'model__C': 0.1, 'prep__num__imputer__strategy': 'mean'}
최적 CV: 0.815
 param_model__C  mean_test_score
           0.01            0.751
           0.10            0.815
           1.00            0.805
          10.00            0.809


**결과** · 예측 정답은 아님 — `0.751`로 가장 낮음, 최적은 `C=0.1` (`0.815`)

- `후보: 8 개 → 최적: {'model__C': 0.1, 'prep__num__imputer__strategy': 'mean'}` — 채우기 방법도 CV로 골라짐
- `최적 CV: 0.815` — train 안에서만 계산 (valid · test는 안 씀)
- `0.01            0.751` — 규제가 너무 강해 과소적합
- `0.10            0.815` — 가장 좋음, 그 뒤로는 비슷

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>모델__설정</code> (<code>model__C</code>)</td>
<td style="text-align: left;">Pipeline 안 모델의 설정</td>
<td style="text-align: left;">모델 하이퍼파라미터 탐색</td>
</tr>
<tr>
<td style="text-align: left;"><code>prep__num__imputer__strategy</code></td>
<td style="text-align: left;">전처리 <strong>안쪽 단계</strong>까지 경로로</td>
<td style="text-align: left;">채우기 방법 · 스케일러 종류도 CV로 고를 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 경로가 길수록 오타가 잦음
> - `pipe.get_params().keys()`로 정확한 이름을 먼저 확인

> ✅ **한 줄 정리** · Pipeline째 GridSearchCV에 넣으면 `이름__이름__설정` 경로로 전처리 설정까지 누수 없이 함께 고른다.

**딥러닝 코드에서는**

- 딥러닝에서는 증강 세기 · 정규화 방식 같은 전처리 선택도 학습률과 똑같이 val 점수로 비교

---
## 5. 딥러닝 연결

> **이 장의 흐름**
>
> - 1 ~ 4장은 sklearn이었다
> - 딥러닝 코드에서도 '정규화 통계는 train으로만', '범주는 train 사전으로 번호'가 그대로 나옴.
> - 이름만 바뀐 같은 원리를 확인

**파이프라인 위치** · ③ 전처리 → ④ 모델 입력 — 이미지 정규화 · 임베딩

### 5-1. `A` train 통계로만 정규화 · `mean · std` <sub>추가 · 원본 03 Part 4 원칙</sub>

> 🎯 **이미지를 채널별 mean · std로 정규화할 때 그 값은 어디서 계산하며, StandardScaler와 무엇이 같은가?**

💡 **쉽게 말하면**

- 이미지 정규화도 **같은 자 쓰기**.
- 빨강 · 초록 · 파랑 채널마다 train 사진들의 평균 · 표준편차를 재 두고, 새 사진도 그 자로 잼.

**실행 흐름**

1. 가짜 이미지 train 20장 · test 5장
2. 0~1
3. train 채널 mean · std
4. 둘 다 정규화
5. StandardScaler로 같은 계산

**먼저 예측** · test 사진을 train의 mean · std로 정규화하면 test 채널 평균은 0이 될까? — 답은 코드 실행 결과 아래에

In [17]:
# 가짜 RGB 이미지: train 20장 · test 5장, (장, 세로, 가로, 채널) uint8
rng_img = np.random.default_rng(0)                                        # 가짜 사진 고정
img_train = rng_img.integers(0, 256, size=(20, 8, 8, 3), dtype=np.uint8)  # train 20장
img_test = rng_img.integers(60, 256, size=(5, 8, 8, 3), dtype=np.uint8)   # 더 밝은 test 5장


# 채널별 평균 · 표준편차를 train에서만 (0~1로 바꾼 뒤, 축 0 · 1 · 2를 접음)
tr01, te01 = img_train / 255, img_test / 255                      # 0 ~ 1로
mean = tr01.mean(axis=(0, 1, 2))                                  # 채널 3개 → (3,)
std = tr01.std(axis=(0, 1, 2))                                    # 채널별 표준편차
print("train 채널 mean:", mean.round(3), "| std:", std.round(3))  # train 통계


# train · test 모두 train의 mean · std로 정규화 (StandardScaler와 같은 공식)
tr_n, te_n = (tr01 - mean) / std, (te01 - mean) / std                      # 같은 통계로 둘 다
print("정규화 후 채널 평균 → train:", tr_n.mean(axis=(0, 1, 2)).round(3),  # 밝기 차이 유지
      "| test:", te_n.mean(axis=(0, 1, 2)).round(3))


# 같은 계산을 StandardScaler로: 픽셀을 행, 채널을 열로 펼침
flat_sc = StandardScaler().fit(tr01.reshape(-1, 3))  # (20·8·8, 3)
print("StandardScaler와 같음:",                      # 같은 공식
      np.allclose(flat_sc.transform(te01.reshape(-1, 3)), te_n.reshape(-1, 3)))

train 채널 mean: [0.497 0.506 0.49 ] | std: [0.289 0.29  0.302]
정규화 후 채널 평균 → train: [-0. -0. -0.] | test: [0.432 0.361 0.407]
StandardScaler와 같음: True


**결과** · 예측 정답은 아님 — `[0.432 0.361 0.407]` (test 사진을 더 밝게 만들었기 때문, 이 차이가 지워지지 않는 것이 정상)

- `train 채널 mean: [0.497 0.506 0.49 ] | std: [0.289 0.29  0.302]` — 축 0 · 1 · 2(장 · 세로 · 가로)를 접고 채널마다
- `정규화 후 채널 평균 → train: [-0. -0. -0.] | test: [0.432 0.361 0.407]` — test가 더 밝다는 정보가 그대로 남음
- `StandardScaler와 같음: True` — 픽셀을 행, 채널을 열로 펼치면 같은 계산

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">train mean · std로 정규화</td>
<td style="text-align: left;">모든 데이터를 <strong>같은 자</strong>로 → test 평균은 0이 아닐 수 있음</td>
<td style="text-align: left;">항상 (StandardScaler와 같은 원리)</td>
</tr>
<tr>
<td style="text-align: left;">데이터마다 자기 mean · std</td>
<td style="text-align: left;">밝기 차이 같은 <strong>진짜 차이를 지움</strong>, 한 장씩 들어오면 계산 불가</td>
<td style="text-align: left;">쓰지 않음</td>
</tr>
<tr>
<td style="text-align: left;">ImageNet mean · std 상수</td>
<td style="text-align: left;">ImageNet <strong>train</strong> 통계 (0.485, 0.456, 0.406 …)</td>
<td style="text-align: left;">사전학습 모델을 가져다 쓸 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - mean · std를 전체(또는 test 포함) 이미지로 계산하면 3-2와 같은 누수
> - 데이터 폴더를 나눈 **뒤** train 폴더로만

> ✅ **한 줄 정리** · 이미지 정규화의 mean · std는 train 이미지로만 계산해 모든 데이터에 같은 값으로 적용하며, 원리는 StandardScaler와 같다.

**딥러닝 코드에서는**

- `transforms.Normalize(mean=train_mean, std=train_std)`
- 사전학습 모델이면 그 모델을 만든 ImageNet train 통계 `mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`

### 5-2. `A` 범주에서 임베딩까지 · `nn.Embedding` <sub>추가 · 원본 03 Part 1 인코딩 확장</sub>

> 🎯 **딥러닝은 범주 · 단어를 어떻게 번호로 바꾸고, 그 번호를 어떻게 벡터로 쓰는가?**

💡 **쉽게 말하면**

- 번호는 **사물함 번호**, 임베딩은 **사물함 안에 든 학습되는 벡터**.
- 번호 자체를 크기로 쓰지 않고(1-2의 함정), 번호로 사물함만 엶.

<img src="images/ml2_embedding.png" alt="원-핫 × W = 그 번호의 행" width="767">

**실행 흐름**

1. train 항구로 사전 (`<unk>` = 0)
2. S · C · X를 번호로
3. nn.Embedding(4, 3)
4. 원-핫 × 가중치와 비교

**먼저 예측** · train에 없던 항구 `X`의 번호는? — 답은 코드 실행 결과 아래에

In [18]:
import torch  # 머신러닝-2에서 유일하게 PyTorch를 쓰는 곳


# 범주 → 번호: train에 나온 범주만 사전에 넣고, 0번은 '처음 보는 값'
train_ports = X_train["Embarked"].fillna("S")     # train 항구만
vocab = {"<unk>": 0}                              # 0번 = 처음 보는 값
for port in sorted(train_ports.unique()):         # 정렬 → 항상 같은 번호
    vocab[port] = len(vocab)                      # 다음 번호 배정
print("사전:", vocab)                             # 완성된 사전
idx = [vocab.get(p, 0) for p in ["S", "C", "X"]]  # X = 처음 보는 항구
print("S · C · X → 번호:", idx)                   # X는 0번


# 번호 → 벡터: 임베딩 = 번호마다 학습되는 벡터 한 줄 (여기서는 3칸)
torch.manual_seed(0)                                                  # 임베딩 초기값 고정
emb = torch.nn.Embedding(num_embeddings=len(vocab), embedding_dim=3)  # 번호마다 3칸 벡터
vecs = emb(torch.tensor(idx))                                         # 번호 3개 → (3, 3)
print("임베딩 결과 shape:", tuple(vecs.shape))                        # (번호 수, 3)


# 원-핫 × 가중치 행렬 = 그 번호의 행을 꺼내기 → 임베딩은 원-핫의 지름길
W = emb.weight.detach()                                                       # (사전 크기, 3)
one_hot = torch.nn.functional.one_hot(torch.tensor(idx), len(vocab)).float()  # 같은 번호의 원-핫
print("원-핫 @ W 와 같음:", torch.allclose(one_hot @ W, vecs.detach()))       # 행 꺼내기 = 곱셈

사전: {'<unk>': 0, 'C': 1, 'Q': 2, 'S': 3}
S · C · X → 번호: [3, 1, 0]
임베딩 결과 shape: (3, 3)
원-핫 @ W 와 같음: True


**결과** · 예측 정답은 `0` — `<unk>` (처음 보는 값을 모으는 칸, OneHotEncoder의 `handle_unknown="ignore"`와 같은 역할)

- `사전: {'<unk>': 0, 'C': 1, 'Q': 2, 'S': 3}` — 정렬해서 번호 고정 (Python 문법 8장과 같은 방법)
- `S · C · X → 번호: [3, 1, 0]` — 사전에 없는 X → 0번
- `임베딩 결과 shape: (3, 3)` — 번호 3개 → 3칸짜리 벡터 3개
- `원-핫 @ W 와 같음: True` — 임베딩 = 원-핫 곱셈을 건너뛴 행 꺼내기

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">원-핫</td>
<td style="text-align: left;">범주 수만큼 열, 대부분 0, 범주끼리 거리가 모두 같음</td>
<td style="text-align: left;">범주가 적은 머신러닝 입력</td>
</tr>
<tr>
<td style="text-align: left;">번호 + <code>nn.Embedding</code></td>
<td style="text-align: left;">번호마다 <strong>학습되는 짧은 벡터</strong>, 비슷한 범주는 가까워질 수 있음</td>
<td style="text-align: left;">범주 · 단어가 많은 딥러닝 입력 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>&lt;unk&gt;</code> = 0번</td>
<td style="text-align: left;">train에 없던 값을 한 칸에 모음</td>
<td style="text-align: left;">사전 만들 때 항상 (<code>handle_unknown="ignore"</code>와 같은 역할)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 사전은 train으로만 만들고 저장
> - valid · test로 사전을 다시 만들면 같은 단어의 번호가 바뀜 (1-1의 `get_dummies` 문제와 같음)

> ✅ **한 줄 정리** · 범주는 train 사전으로 번호를 매기고(`<unk>` 포함), 딥러닝에서는 그 번호로 `nn.Embedding`의 학습되는 벡터를 꺼내 쓴다.

**딥러닝 코드에서는**

- 텍스트 모델의 첫 줄
- `tokens → vocab 번호 → nn.Embedding(vocab_size, dim)` → RNN · Transformer 입력 `(배치, 길이, dim)`

---
## 6. 종합 실습 · 타이타닉 전처리 파이프라인

> **이 장의 흐름** · 1 ~ 5장을 타이타닉 표 하나에 적용. 미션 3 정답1의 `preprocessing(scale_numeric)` · `model_pipe` 구조를 함수 하나로 정리하고, 4-1 로지스틱 Part 2의 3분할(test 봉인)을 그대로 이어서 씀

**파이프라인 위치** · ③ 전처리 → ④ 모델 → ⑤ 교차검증 비교 → ⑥ valid 1회 (test는 머신러닝-3 종합 실습)

> 🎯 **모델마다 필요한 전처리가 다를 때, 열마다 다른 전처리를 묶어 누수 없이 비교 · 확인하려면?**

- 💡 **쉽게 말하면** 같은 재료(타이타닉 표)라도 **요리(모델)마다 손질법이 다름**.
- 거리 · 가중합 모델(KNN · 로지스틱)은 단위를 맞추고, 트리는 맞출 필요가 없음.
- 손질 방법을 스위치 하나로 바꾸는 함수를 만들어 같은 5등분으로 비교함.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰는 섹션</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;"><code>build(model, scale)</code> · 숫자 열(채우기 → 선택적 표준화) + 글자 열(채우기 → 원-핫)</td>
<td style="text-align: left;">4-2 · 4-3 · 2-1</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">train 534명 안에서 LR · KNN(스케일 O · X) · 트리를 5-Fold 비교</td>
<td style="text-align: left;">2-3 · 3-2</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">누수 점검 · Pipeline 밖에서 미리 채우기 · 표준화한 경우와 비교</td>
<td style="text-align: left;">3-2 · 3-3</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">고른 LR 묶음을 train 전체로 fit → valid 1회 확인 (test 봉인 유지)</td>
<td style="text-align: left;">3-1 · 머신러닝-1 5-2</td>
</tr>
</tbody>
</table>

**먼저 예측** · KNN은 스케일링을 빼면 CV가 얼마나 떨어질까? 트리는 스케일링 없이도 버틸까? — 답은 코드 실행 결과 아래에

In [19]:
# 1단계: 모델에 맞는 전처리 묶음을 만드는 함수 (미션 3 정답1의 구조)
def build(model, scale):                                              # 모델별 전처리 묶음 만들기
    num_steps = [("imputer", SimpleImputer(strategy="median"))]       # 숫자: 채우기는 항상
    if scale:                                                         # 거리 · 가중합 모델만 표준화
        num_steps.append(("scaler", StandardScaler()))                # 거리 · 가중합 모델만
    cat = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),  # 글자: 채우기 → 원-핫
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    pre = ColumnTransformer([("num", Pipeline(num_steps), num_cols),  # 열 종류별 라인
                             ("cat", cat, cat_cols)])
    return Pipeline([("prep", pre), ("model", model)])                # 전처리 + 모델


# 2단계: train 534명 안에서만 5-Fold로 비교 (valid · test는 안 씀)
cands = [("LR", LogisticRegression(max_iter=5000), True),               # (이름, 모델, 스케일 여부)
         ("KNN", KNeighborsClassifier(n_neighbors=5), True),
         ("KNN", KNeighborsClassifier(n_neighbors=5), False),
         ("Tree", DecisionTreeClassifier(max_depth=4, random_state=42), False)]
for name, model, scale in cands:                                        # 후보마다
    s = cross_val_score(build(model, scale), X_train, y_train, cv=cv5)  # Pipeline째 CV
    tag = "스케일 O" if scale else "스케일 X"                           # 출력용 표시
    print(f"{name:<4} {tag} CV {s.mean():.3f} ± {s.std():.3f}")         # 평균 ± 흔들림

LR   스케일 O CV 0.805 ± 0.041


KNN  스케일 O CV 0.803 ± 0.056
KNN  스케일 X CV 0.714 ± 0.047
Tree 스케일 X CV 0.792 ± 0.047


In [20]:
# 3단계: 누수 점검 → Pipeline 밖에서 train 전체로 미리 채우기 · 표준화한 경우
leak_X = X_train.copy()                                               # 원본은 그대로 두고
imp_all = SimpleImputer(strategy="median")                            # train 534명 전체로 fit
leak_X[num_cols] = imp_all.fit_transform(leak_X[num_cols])            # 전체로 채우기 (누수)
leak_X[num_cols] = StandardScaler().fit_transform(leak_X[num_cols])   # 전체로 표준화 (누수)
leak_pipe = build(LogisticRegression(max_iter=5000), scale=False)     # 스케일은 이미 함
leak = cross_val_score(leak_pipe, leak_X, y_train, cv=cv5).mean()     # 누수 방식 CV
ok = cross_val_score(build(LogisticRegression(max_iter=5000), True),  # Pipeline 방식 CV
                     X_train, y_train, cv=cv5).mean()
print(f"LR 누수 CV {leak:.3f} | Pipeline CV {ok:.3f}")                # 점수가 같아도 절차 위반


# 4단계: 고른 LR 묶음을 train 전체로 fit → valid로 1회 확인 (test는 봉인)
final_t = build(LogisticRegression(max_iter=5000), True).fit(X_train, y_train)    # train 전체로
print(f"valid 정확도: {final_t.score(X_valid, y_valid):.3f}")                     # valid 1회 확인
print("변환 후 열 수:", final_t.named_steps["prep"].transform(X_valid).shape[1])  # 19열
print("test 행 수:", len(X_test), "→ 머신러닝-3 종합 실습에서 1회 사용")          # test는 봉인

LR 누수 CV 0.805 | Pipeline CV 0.805
valid 정확도: 0.798
변환 후 열 수: 19
test 행 수: 179 → 머신러닝-3 종합 실습에서 1회 사용


**결과** · 예측 정답은 KNN `0.803` → `0.714` (약 0.09 하락) · 트리는 스케일링 없이 `0.792`

- `LR   스케일 O CV 0.805 ± 0.041` — 가장 높음 → 최종 후보
- `KNN  스케일 O CV 0.803 ± 0.056` — LR과 비슷하지만 흔들림(std)이 더 큼
- `KNN  스케일 X CV 0.714 ± 0.047` — 스케일링을 빼면 크게 하락 (2-3 펭귄과 같은 현상)
- `Tree 스케일 X CV 0.792 ± 0.047` — 트리는 기준값만 보므로 스케일링이 필요 없음
- `LR 누수 CV 0.805 | Pipeline CV 0.805` — 점수가 같아도 누수 방식은 절차 위반 (3-2와 같은 결론)
- `valid 정확도: 0.798` — 4-1 로지스틱 원본의 `기본 Accuracy 0.798`과 같은 값
- `변환 후 열 수: 19` — 4-3과 같은 19열
- `test 행 수: 179 → 머신러닝-3 종합 실습에서 1회 사용` — test는 아직 한 번도 쓰지 않음

<img src="images/ml2_result.png" alt="모델별 전처리 비교 결과" width="632">

**해석**

- LR과 KNN의 CV 차이(0.002)는 머신러닝-1 3-5 기준으로 우열이 아님
- std가 작고 확률을 내는 LR을 고름 · valid 0.798이 CV 0.805와 비슷해 과적합 신호도 없음
- 확률 · 임곗값 · 최종 test는 머신러닝-3에서 이 `final_t`와 같은 묶음으로 이어 감

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">모든 모델에 같은 전처리</td>
<td style="text-align: left;">트리에도 표준화 → 해롭지는 않지만 불필요</td>
<td style="text-align: left;">모델이 하나뿐일 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>build(model, scale)</code> 함수</td>
<td style="text-align: left;">스위치 하나로 모델별 전처리, 같은 열 목록 재사용</td>
<td style="text-align: left;">여러 모델을 공정하게 비교할 때 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;">Pipeline 밖에서 미리 전처리</td>
<td style="text-align: left;">fold의 검증 부분까지 섞인 기준</td>
<td style="text-align: left;">쓰지 않음</td>
</tr>
<tr>
<td style="text-align: left;">Pipeline째 CV</td>
<td style="text-align: left;">fold마다 학습 부분으로만 fit</td>
<td style="text-align: left;">항상</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `leak_X`처럼 미리 변환한 표를 만들면 이후 코드에서 원래 `X_train`과 섞어 쓰기 쉬움
> - 변환은 Pipeline 안에서만, 표는 원본 그대로 들고 다니기

> ✅ **한 줄 정리** · 모델별 전처리를 함수 하나의 Pipeline으로 만들고, train 안에서 Pipeline째 교차검증해 고른 뒤 valid로 한 번 확인하며 test는 봉인한다.

**딥러닝 코드에서는**

- `Dataset` 안의 `transform`(정규화 · 사전 번호)은 train 통계로 만든 하나를 train · val · test loader가 함께 씀
- 모델별로 다른 입력 전처리는 `transform`만 바꿔 끼움

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">학습 목표</th>
<th style="text-align: left;">할 수 있게 된 것</th>
<th style="text-align: left;">확인 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1. 인코더 고르기</td>
<td style="text-align: left;">순서 없는 X는 원-핫, 순서 있는 X는 Ordinal, y는 Label · 번호를 선형 모델에 넣으면 0.640</td>
<td style="text-align: left;">1-1 ~ 1-3 · 선택표</td>
</tr>
<tr>
<td style="text-align: left;">2. 스케일러 고르기</td>
<td style="text-align: left;">이상치에 Robust, 픽셀에 MinMax, 기본 Standard · KNN 0.784 → 0.982</td>
<td style="text-align: left;">2-1 ~ 2-3 · 선택표</td>
</tr>
<tr>
<td style="text-align: left;">3. fit · transform · 누수</td>
<td style="text-align: left;">자는 train으로만 · 누수는 점수가 아니라 절차로 판단 · 난수에서 0.870 vs 0.520</td>
<td style="text-align: left;">3-1 ~ 3-3</td>
</tr>
<tr>
<td style="text-align: left;">4. Pipeline으로 묶기</td>
<td style="text-align: left;">SimpleImputer · ColumnTransformer로 8열 → 19열 · <code>prep__num__imputer__strategy</code> 탐색</td>
<td style="text-align: left;">4-1 ~ 4-4</td>
</tr>
<tr>
<td style="text-align: left;">5. 딥러닝 연결</td>
<td style="text-align: left;">train 채널 mean · std로 정규화 · train 사전(<code>&lt;unk&gt;</code> = 0) → <code>nn.Embedding</code></td>
<td style="text-align: left;">5-1 · 5-2</td>
</tr>
</tbody>
</table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: center;">등급</th>
<th style="text-align: left;">주제</th>
<th style="text-align: left;">반드시 기억할 것</th>
<th style="text-align: left;">딥러닝 코드에서</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">원-핫</td>
<td style="text-align: left;">순서 없는 X는 <code>OneHotEncoder(handle_unknown="ignore")</code></td>
<td style="text-align: left;">범주가 많으면 임베딩</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">가짜 순서</td>
<td style="text-align: left;">번호를 선형 모델에 넣으면 크기로 믿음</td>
<td style="text-align: left;">번호는 찾아보기 번호로만</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">Ordinal · Label</td>
<td style="text-align: left;">순서 있는 X는 Ordinal, y는 Label</td>
<td style="text-align: left;"><code>classes_</code> 저장 → 예측 번호를 이름으로</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">StandardScaler</td>
<td style="text-align: left;">열마다 평균 0 · 표준편차 1</td>
<td style="text-align: left;">표 입력 정규화</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">스케일러 3종</td>
<td style="text-align: left;">기본 Standard, 이상치 Robust, 픽셀 MinMax</td>
<td style="text-align: left;"><code>/ 255</code> → mean · std</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">스케일링 효과</td>
<td style="text-align: left;">거리 · 가중합 모델은 필수, 트리는 불필요</td>
<td style="text-align: left;">입력 단위부터 확인</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">fit · transform</td>
<td style="text-align: left;">fit은 train에만, transform은 전부</td>
<td style="text-align: left;">scaler · 사전 저장</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">누수</td>
<td style="text-align: left;">점수가 아니라 절차로 판단, Pipeline째 CV</td>
<td style="text-align: left;">통계 · 사전은 train 폴더로만</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">선택 누수</td>
<td style="text-align: left;">정답을 보는 선택도 train 안에서</td>
<td style="text-align: left;">val로 고르고 test 1회</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">Pipeline</td>
<td style="text-align: left;"><code>이름__설정</code>, <code>named_steps</code></td>
<td style="text-align: left;"><code>nn.Sequential</code> · <code>Compose</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">SimpleImputer</td>
<td style="text-align: left;">train 중앙값 · 최빈값을 기억</td>
<td style="text-align: left;">train 구간으로 채우기</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">ColumnTransformer</td>
<td style="text-align: left;">열 종류별 Pipeline → 이어 붙이기</td>
<td style="text-align: left;">입력 종류별 전처리 후 concat</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">이미지 mean · std</td>
<td style="text-align: left;">train 채널 통계로 모두 정규화</td>
<td style="text-align: left;"><code>transforms.Normalize</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">임베딩</td>
<td style="text-align: left;">train 사전(<code>&lt;unk&gt;</code> = 0) → <code>nn.Embedding</code></td>
<td style="text-align: left;">텍스트 모델 첫 줄</td>
</tr>
<tr>
<td style="text-align: center;"><code>B</code></td>
<td style="text-align: left;">CountVectorizer</td>
<td style="text-align: left;">train 사전으로 단어 개수, 새 문장은 transform</td>
<td style="text-align: left;">토크나이저 · vocab</td>
</tr>
<tr>
<td style="text-align: center;"><code>B</code></td>
<td style="text-align: left;">정규화</td>
<td style="text-align: left;">행 길이 1 → 내적 = 코사인</td>
<td style="text-align: left;"><code>F.normalize</code> 후 내적</td>
</tr>
</tbody>
</table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">🎯 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;"><code>OneHotEncoder</code>와 <code>get_dummies</code>는 나중에 들어온 데이터에서 어떻게 다른가?</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">순서 없는 범주에 번호를 붙여 로지스틱에 넣으면 왜 문제인가?</td>
</tr>
<tr>
<td style="text-align: left;">1-3</td>
<td style="text-align: left;"><code>OrdinalEncoder</code>와 <code>LabelEncoder</code>는 각각 어디에 쓰는가?</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">단위가 다른 특성은 거리 계산에서 어떤 문제를 만드는가?</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">이상치가 있을 때 고를 스케일러는?</td>
</tr>
<tr>
<td style="text-align: left;">2-4</td>
<td style="text-align: left;">스케일링과 정규화는 무엇을 기준으로 맞추는가?</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;"><code>fit</code> · <code>transform</code> · <code>fit_transform</code>을 train · test에 어떻게 나눠 쓰는가?</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;">누수 방식과 Pipeline의 점수가 같으면 누수 방식도 괜찮은가?</td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;">정답을 보고 특성을 고르는 전처리는 왜 위험한가?</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;">Pipeline 안 모델의 설정을 GridSearchCV에 넘기는 이름 형식은?</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">결측을 채울 중앙값은 어느 데이터로 계산하는가?</td>
</tr>
<tr>
<td style="text-align: left;">4-3</td>
<td style="text-align: left;">숫자 · 글자 열에 다른 전처리를 한 번에 하려면?</td>
</tr>
<tr>
<td style="text-align: left;">5-1</td>
<td style="text-align: left;">이미지 정규화의 mean · std는 어디서 계산하는가?</td>
</tr>
<tr>
<td style="text-align: left;">5-2</td>
<td style="text-align: left;">범주를 딥러닝 모델에 넣는 순서는?</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">모델마다 전처리가 다를 때 공정하게 비교하는 방법은?</td>
</tr>
</tbody>
</table>

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

원본 노트북의 예제를 조건만 바꿔 만든 문제. 아래 빈 셀에 직접 작성 후 실행.

1. `["M", "S", "XL", "L", "S"]`를 **S < M < L < XL** 순서의 번호로 바꾸기 (1-3)
2. `x = [[10], [20], [40]]`을 **공식**으로 0 ~ 1로 바꾸고 `MinMaxScaler` 결과와 같은지 확인 (2-2)
3. train `["red", "blue", "red"]`로 원-핫 기준을 만들고, test `["blue", "green"]`을 **열 수가 유지되게** 변환 (1-1)
4. 타이타닉에서 `Fare`는 `RobustScaler`, `Sex`는 원-핫으로 바꾸는 `ColumnTransformer`를 **앞 600명으로 fit**하고 나머지를 변환한 shape (4-3)
5. 아래 누수 코드를 Pipeline으로 고쳐 붓꽃 5-Fold(`cv5`) 평균 출력 (3-2)

```python
X_s = StandardScaler().fit_transform(X)  # 전체로 fit → 누수
cross_val_score(KNeighborsClassifier(n_neighbors=5), X_s, y, cv=cv5)
```

In [21]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 2-2의 이상치 100을 **1000**으로 바꾸면 MinMax · Robust의 정상 1 → 3 간격은 각각 얼마가 될까? 예측 → 실행
2. 4-3의 `cat_cols`에서 `Cabin1`을 빼면 변환 후 열 수는 몇 개일까?
3. 3-1의 분할을 `test_size=0.5`로 바꿔도 "test에 따로 fit"이 train 기준보다 높게 나올까?

In [22]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;"><code>OneHotEncoder</code>는 fit한 범주를 기억해 같은 열을 유지(모르는 값은 <code>handle_unknown="ignore"</code>로 전부 0), <code>get_dummies</code>는 매번 새로 만들어 열이 달라짐</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">선형 모델이 번호를 크기로 믿어 가짜 순서가 생김 (가운데 번호만 0인 규칙을 표현 못 해 0.640)</td>
</tr>
<tr>
<td style="text-align: left;">1-3</td>
<td style="text-align: left;">Ordinal은 진짜 순서가 있는 X(순서 지정), Label은 정답 y (알파벳 순서)</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">단위가 큰 특성이 거리를 지배 (용돈 몫 100%) → StandardScaler</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">RobustScaler (중앙값 · IQR 기준)</td>
</tr>
<tr>
<td style="text-align: left;">2-4</td>
<td style="text-align: left;">스케일링은 열(특성)마다, 정규화는 행(샘플)마다 길이 1</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;">fit · fit_transform은 train에만, transform은 train · valid · test 모두</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;">아님. 누수는 점수 차이가 아니라 절차로 판단</td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;">정답 정보가 새어 들어가 없는 패턴을 만듦 (난수에서 0.870)</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;"><code>단계이름__설정</code> (밑줄 2개, 예: <code>knn__n_neighbors</code>)</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">train (전체로 계산하면 test 정보가 섞임)</td>
</tr>
<tr>
<td style="text-align: left;">4-3</td>
<td style="text-align: left;"><code>ColumnTransformer</code>에 열 목록별 Pipeline 지정</td>
</tr>
<tr>
<td style="text-align: left;">5-1</td>
<td style="text-align: left;">train 이미지로만 채널별 계산 → 모든 데이터에 같은 값 적용</td>
</tr>
<tr>
<td style="text-align: left;">5-2</td>
<td style="text-align: left;">train 사전으로 번호(<code>&lt;unk&gt;</code> = 0) → <code>nn.Embedding</code>으로 벡터</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">모델별 전처리를 Pipeline 함수로 만들고 같은 5등분으로 Pipeline째 CV</td>
</tr>
</tbody>
</table>

#### 코드 문제 답

```python
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_iris
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    MinMaxScaler, OneHotEncoder, OrdinalEncoder, RobustScaler,
    StandardScaler)

# 1. 옷 크기 → 순서 지정 (S < M < L < XL)
shirts = pd.DataFrame({"size": ["M", "S", "XL", "L", "S"]})
enc = OrdinalEncoder(categories=[["S", "M", "L", "XL"]])
codes = enc.fit_transform(shirts[["size"]]).ravel().tolist()
print(codes)  # [1.0, 0.0, 3.0, 2.0, 0.0]

# 2. MinMax 공식 = (x - 최솟값) / (최댓값 - 최솟값)
x = np.array([[10.], [20.], [40.]])
manual = (x - x.min()) / (x.max() - x.min())
print(manual.ravel().round(3).tolist())  # [0.0, 0.333, 1.0]
print(np.allclose(manual, MinMaxScaler().fit_transform(x)))  # True

# 3. train 기준 원-핫 → test에 처음 보는 범주가 있어도 열 수 유지
train = pd.DataFrame({"color": ["red", "blue", "red"]})
test = pd.DataFrame({"color": ["blue", "green"]})
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(train)
print(ohe.get_feature_names_out().tolist())  # ['color_blue', 'color_red']
print(ohe.transform(test).tolist())  # [[1.0, 0.0], [0.0, 0.0]]

# 4. 타이타닉: Fare는 RobustScaler, Sex는 원-핫 → 앞 600명으로 fit
titanic = pd.read_csv("train.csv")
ct = ColumnTransformer([("fare", RobustScaler(), ["Fare"]),
                        ("sex", OneHotEncoder(), ["Sex"])])
ct.fit(titanic.iloc[:600])
print(ct.transform(titanic.iloc[600:]).shape)  # (291, 3)

# 5. 누수 고치기: scaler를 Pipeline 안으로
X, y = load_iris(return_X_y=True)
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe = Pipeline([("scaler", StandardScaler()),
                 ("knn", KNeighborsClassifier(n_neighbors=5))])
print(round(cross_val_score(pipe, X, y, cv=cv5).mean(), 3))  # 0.973
```

**5번 확인** · 3-2의 Pipeline 점수 `0.973`과 같음 (누수 방식은 0.967)


#### 변형 문제 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">문제</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">MinMax 0.002 · Robust 0.008 → 샘플이 4개뿐이면 Robust의 IQR도 이상치에 끌려감 (100일 때 0.078)</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">19 - 9 = 10열 (숫자 5 + Sex 2 + Embarked 3)</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">아님 · train 기준 0.947 vs 따로 fit 0.920 → 7 : 3에서 높게 나온 건 우연, 절차가 틀린 것은 그대로</td>
</tr>
</tbody>
</table>

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">틀린 내용</th>
<th style="text-align: left;">틀린 이유</th>
<th style="text-align: left;">올바른 원칙</th>
<th style="text-align: left;">재발 방지 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">(예) 전체 데이터로 scaler를 fit하고 교차검증함</td>
<td style="text-align: left;">점수가 비슷해서 괜찮다고 판단</td>
<td style="text-align: left;">누수는 절차로 판단, 전처리는 Pipeline 안에서</td>
<td style="text-align: left;">이 fit 괄호 안에 검증 데이터가 들어 있나?</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
</tbody>
</table>

---
## 면접 30초 답변

**Q1. 원-핫 인코딩과 레이블 인코딩은 언제 쓰나요?**
> - 레이블 인코딩은 범주에 0, 1, 2 번호를 붙여 선형 모델이 그 번호를 크기로 오해할 수 있어 주로 정답 y에 씁니다.
> - 순서가 없는 입력 범주는 원-핫으로 범주마다 0/1 열을 만들고, 크기처럼 진짜 순서가 있으면 OrdinalEncoder로 순서를 지정합니다.
> - 모델 입력에는 train 범주를 기억하는 OneHotEncoder를 써서 새 데이터에서도 같은 열이 나오게 합니다.

**Q2. 데이터 누수란 무엇이고 어떻게 막나요?**
> - 평가용으로 남겨 둔 데이터의 정보가 학습 과정에 들어가는 것입니다.
> - 전체 데이터로 스케일러를 맞추거나 정답을 보고 특성을 고르면, 점수가 같아 보여도 평가를 믿을 수 없고 난수 데이터에서 87%처럼 없는 성능이 나오기도 합니다.
> - 전처리를 Pipeline에 넣고 Pipeline째 교차검증해 fold마다 학습 부분으로만 fit하게 합니다.

**Q3. StandardScaler, MinMaxScaler, RobustScaler는 어떻게 고르나요?**
> - 기본은 평균 0, 표준편차 1로 맞추는 StandardScaler입니다.
> - 이상치가 있으면 중앙값과 사분위 범위를 쓰는 RobustScaler가 정상값의 구분을 지키고, 픽셀처럼 범위가 정해진 값은 MinMaxScaler가 자연스럽습니다.
> - 트리는 스케일링이 필요 없고, 후보는 같은 교차검증으로 비교해 고릅니다.

**Q4. 이미지 정규화의 mean과 std는 어디서 구하나요?**
> - train 이미지로만 채널별 평균과 표준편차를 계산해 train, validation, test에 모두 같은 값으로 적용합니다.
> - StandardScaler를 train에만 fit하는 것과 같은 원리이고, 사전학습 모델을 쓸 때는 그 모델이 학습한 ImageNet train 통계를 그대로 씁니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
<th style="text-align: left;">코드에서 사용한 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>LabelEncoder()</code> · <code>classes_</code> · <code>inverse_transform()</code></td>
<td style="text-align: left;">sklearn.preprocessing</td>
<td style="text-align: left;">정답 y 번호 ↔ 이름</td>
<td style="text-align: left;">1-1 <code>le.fit_transform(data["color"])</code> · 1-3 <code>le_y.inverse_transform(codes)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>OrdinalEncoder(categories=)</code></td>
<td style="text-align: left;">sklearn.preprocessing</td>
<td style="text-align: left;">순서를 지정한 번호</td>
<td style="text-align: left;">1-3 <code>OrdinalEncoder(categories=order).fit_transform(</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>OneHotEncoder(sparse_output=, handle_unknown=)</code></td>
<td style="text-align: left;">sklearn.preprocessing</td>
<td style="text-align: left;">범주마다 0/1 열, 모르는 값 처리</td>
<td style="text-align: left;">1-1 <code>OneHotEncoder(sparse_output=False, handle_unknown="ignore")</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>get_feature_names_out()</code></td>
<td style="text-align: left;">변환기 메서드</td>
<td style="text-align: left;">변환 후 열 이름</td>
<td style="text-align: left;">1-1 <code>print("OneHot 열:", ohe.get_feature_names_out().tol…</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>pd.get_dummies()</code></td>
<td style="text-align: left;">pandas</td>
<td style="text-align: left;">0/1 표 바로 만들기 (기준 기억 X)</td>
<td style="text-align: left;">1-1 <code>dummies = pd.get_dummies(data["color"], dtype=int)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>CountVectorizer()</code></td>
<td style="text-align: left;">sklearn.feature_extraction.text</td>
<td style="text-align: left;">문장 → 단어 개수 벡터</td>
<td style="text-align: left;">1-4 <code>vec = CountVectorizer()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>predict_proba()</code></td>
<td style="text-align: left;">분류 모델 메서드</td>
<td style="text-align: left;">클래스별 확률 (머신러닝-3)</td>
<td style="text-align: left;">1-2 <code>lr_label.predict_proba([[v]])[0, 1]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>StandardScaler()</code> · <code>mean_</code> · <code>scale_</code></td>
<td style="text-align: left;">sklearn.preprocessing</td>
<td style="text-align: left;">평균 0 · 표준편차 1, 저장된 기준</td>
<td style="text-align: left;">2-1 <code>scaler = StandardScaler().fit(X_hm)</code> · <code>scaler.mean_</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>MinMaxScaler()</code></td>
<td style="text-align: left;">sklearn.preprocessing</td>
<td style="text-align: left;">최솟값 0 · 최댓값 1</td>
<td style="text-align: left;">2-2 <code>scalers = {"Standard": StandardScaler(), "MinMax":…</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>RobustScaler()</code></td>
<td style="text-align: left;">sklearn.preprocessing</td>
<td style="text-align: left;">중앙값 · IQR 기준</td>
<td style="text-align: left;">2-2 <code>"Robust": RobustScaler()}</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>Normalizer()</code></td>
<td style="text-align: left;">sklearn.preprocessing</td>
<td style="text-align: left;">행 길이 1</td>
<td style="text-align: left;">2-4 <code>X_unit = Normalizer().fit_transform(X_vec)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>cosine_similarity()</code></td>
<td style="text-align: left;">sklearn.metrics.pairwise</td>
<td style="text-align: left;">방향 유사도</td>
<td style="text-align: left;">2-4 <code>cosine_similarity(A, B)[0, 0]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>np.linalg.norm()</code> · <code>@</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">벡터 길이 · 내적</td>
<td style="text-align: left;">2-4 <code>print("행 길이:", np.linalg.norm(X_unit, axis=1).roun…</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>fit()</code> · <code>transform()</code> · <code>fit_transform()</code></td>
<td style="text-align: left;">모든 sklearn 변환기</td>
<td style="text-align: left;">기준 계산 · 적용 · 둘 다</td>
<td style="text-align: left;">3-1 <code>sc = StandardScaler().fit(X_tr)</code> · <code>sc.transform(X_te)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>Pipeline([(이름, 단계), ...])</code></td>
<td style="text-align: left;">sklearn.pipeline</td>
<td style="text-align: left;">전처리 + 모델 묶음</td>
<td style="text-align: left;">3-2 <code>Pipeline([("scaler", StandardScaler()), ("model", model)])</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>cross_validate(return_estimator=True)</code></td>
<td style="text-align: left;">sklearn.model_selection</td>
<td style="text-align: left;">fold별 학습된 모델 꺼내기</td>
<td style="text-align: left;">3-2 <code>cross_validate(…, return_estimator=True)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>SelectKBest(f_classif, k=)</code></td>
<td style="text-align: left;">sklearn.feature_selection</td>
<td style="text-align: left;">y와 관련 높아 보이는 열 k개 선택</td>
<td style="text-align: left;">3-3 <code>X_sel = SelectKBest(f_classif, k=20).fit_transform…</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>named_steps[이름]</code> · <code>이름__설정</code></td>
<td style="text-align: left;">Pipeline</td>
<td style="text-align: left;">단계 꺼내기 · 안쪽 설정 지정</td>
<td style="text-align: left;">4-1 <code>pipe_knn.named_steps["scaler"]</code> · <code>{"knn__n_neighbors": ks}</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>SimpleImputer(strategy=)</code> · <code>statistics_</code></td>
<td style="text-align: left;">sklearn.impute</td>
<td style="text-align: left;">결측 채우기 · 채울 값 기억</td>
<td style="text-align: left;">4-2 <code>SimpleImputer(strategy="median").fit(X_train[["Age"]])</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>ColumnTransformer([(이름, 변환, 열 목록)], remainder=)</code></td>
<td style="text-align: left;">sklearn.compose</td>
<td style="text-align: left;">열마다 다른 전처리</td>
<td style="text-align: left;">4-3 <code>prep = ColumnTransformer([("num", num_pipe, num_cols),</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>str[:1]</code> · <code>fillna()</code></td>
<td style="text-align: left;">pandas</td>
<td style="text-align: left;">첫 글자 · 결측을 범주 N으로</td>
<td style="text-align: left;">4-2 <code>X_t["Cabin"].fillna("N").str[:1]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>GridSearchCV(pipe, {이름__설정: [...]})</code></td>
<td style="text-align: left;">sklearn.model_selection</td>
<td style="text-align: left;">전처리 + 모델 설정 동시 탐색</td>
<td style="text-align: left;">4-4 <code>GridSearchCV(lr_pipe, space, cv=cv5)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>arr.mean(axis=(0, 1, 2))</code> · <code>std()</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">채널별 평균 · 표준편차</td>
<td style="text-align: left;">5-1 <code>tr01.mean(axis=(0, 1, 2))</code> · <code>tr01.std(axis=(0, 1, 2))</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>transforms.Normalize(mean, std)</code></td>
<td style="text-align: left;">torchvision (설명만)</td>
<td style="text-align: left;">이미지 채널 정규화</td>
<td style="text-align: left;">— (설명만 · 5-1 딥러닝 코드에서는)</td>
</tr>
<tr>
<td style="text-align: left;"><code>{"&lt;unk&gt;": 0, ...}</code> · <code>dict.get(키, 0)</code></td>
<td style="text-align: left;">Python 기본 문법</td>
<td style="text-align: left;">범주 사전 · 모르는 값 0번</td>
<td style="text-align: left;">5-2 <code>vocab = {"&lt;unk&gt;": 0}</code> · <code>vocab.get(p, 0)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>nn.Embedding(개수, 차원)</code> · <code>.weight</code></td>
<td style="text-align: left;">torch.nn</td>
<td style="text-align: left;">번호 → 학습되는 벡터</td>
<td style="text-align: left;">5-2 <code>torch.nn.Embedding(num_embeddings=len(vocab), …)</code> · <code>emb.weight</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>F.one_hot()</code></td>
<td style="text-align: left;">torch.nn.functional</td>
<td style="text-align: left;">번호 → 원-핫</td>
<td style="text-align: left;">5-2 <code>torch.nn.functional.one_hot(torch.tensor(idx), len(vocab))</code></td>
</tr>
</tbody>
</table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">✅ 할 것</th>
<th style="text-align: left;">⚠️ 하지 말 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">전처리는 Pipeline 안에서 fit</td>
<td style="text-align: left;">전체 데이터로 scaler · imputer를 먼저 fit</td>
</tr>
<tr>
<td style="text-align: left;">모델 입력 원-핫은 <code>OneHotEncoder</code></td>
<td style="text-align: left;">train · test에 <code>get_dummies</code>를 따로</td>
</tr>
<tr>
<td style="text-align: left;">변환 후 열 수 · 빈칸 확인</td>
<td style="text-align: left;">순서 없는 범주에 번호만 붙여 선형 모델에</td>
</tr>
<tr>
<td style="text-align: left;">test도 반드시 <code>transform</code></td>
<td style="text-align: left;">test에 따로 <code>fit_transform</code></td>
</tr>
<tr>
<td style="text-align: left;">정답을 보는 선택도 CV 안에서</td>
<td style="text-align: left;">점수가 같다고 누수를 괜찮다고 판단</td>
</tr>
<tr>
<td style="text-align: left;">정규화 통계 · 사전은 train으로만 · 저장</td>
<td style="text-align: left;">valid · test로 사전을 다시 만들기</td>
</tr>
</tbody>
</table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">짝</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">결론</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>LabelEncoder</code> vs <code>OrdinalEncoder</code></td>
<td style="text-align: left;">알파벳 순서 · 1차원(y용) vs 지정 순서 · 2차원(X용)</td>
<td style="text-align: left;">y는 Label, 순서 있는 X는 Ordinal</td>
</tr>
<tr>
<td style="text-align: left;"><code>OneHotEncoder</code> vs <code>get_dummies</code></td>
<td style="text-align: left;">범주 기억 vs 매번 새로</td>
<td style="text-align: left;">모델 입력은 <code>OneHotEncoder</code></td>
</tr>
<tr>
<td style="text-align: left;">번호 vs 원-핫 (선형 모델)</td>
<td style="text-align: left;">가짜 크기 생김 vs 범주마다 가중치</td>
<td style="text-align: left;">순서 없는 X는 원-핫</td>
</tr>
<tr>
<td style="text-align: left;"><code>StandardScaler</code> vs <code>MinMaxScaler</code> vs <code>RobustScaler</code></td>
<td style="text-align: left;">평균 · 표준편차 vs 최솟값 · 최댓값 vs 중앙값 · IQR</td>
<td style="text-align: left;">기본 Standard, 이상치면 Robust</td>
</tr>
<tr>
<td style="text-align: left;">스케일링 vs 정규화</td>
<td style="text-align: left;">열마다 vs 행마다</td>
<td style="text-align: left;">단위 차이 vs 방향 비교</td>
</tr>
<tr>
<td style="text-align: left;"><code>fit</code> vs <code>transform</code> vs <code>fit_transform</code></td>
<td style="text-align: left;">기준 계산 vs 변환 vs 둘 다</td>
<td style="text-align: left;">fit은 train에만</td>
</tr>
<tr>
<td style="text-align: left;">누수 전처리 vs <code>Pipeline</code></td>
<td style="text-align: left;">전체로 fit vs fold마다 fit</td>
<td style="text-align: left;">항상 Pipeline</td>
</tr>
<tr>
<td style="text-align: left;">점수 차이 vs 절차</td>
<td style="text-align: left;">누수인지 판단 기준</td>
<td style="text-align: left;">절차로 판단</td>
</tr>
<tr>
<td style="text-align: left;"><code>fillna</code> vs <code>SimpleImputer</code></td>
<td style="text-align: left;">값 직접 보관 vs <code>statistics_</code>에 기억</td>
<td style="text-align: left;">모델 입력은 Imputer</td>
</tr>
<tr>
<td style="text-align: left;"><code>Pipeline</code> vs <code>ColumnTransformer</code></td>
<td style="text-align: left;">단계를 세로로 잇기 vs 열마다 가로로 나누기</td>
<td style="text-align: left;">둘을 겹쳐 씀</td>
</tr>
<tr>
<td style="text-align: left;"><code>model__C</code> vs <code>prep__num__imputer__strategy</code></td>
<td style="text-align: left;">모델 설정 vs 전처리 안쪽 설정</td>
<td style="text-align: left;">경로를 밑줄 2개로</td>
</tr>
<tr>
<td style="text-align: left;">train 통계 vs 데이터마다 통계</td>
<td style="text-align: left;">같은 자 vs 다른 자</td>
<td style="text-align: left;">train 통계</td>
</tr>
<tr>
<td style="text-align: left;">원-핫 vs 임베딩</td>
<td style="text-align: left;">고정된 0/1 vs 학습되는 벡터</td>
<td style="text-align: left;">범주가 많으면 임베딩</td>
</tr>
</tbody>
</table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">언제</th>
<th style="text-align: left;">할 일</th>
<th style="text-align: left;">걸리는 시간</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">내일</td>
<td style="text-align: left;">기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td>
<td style="text-align: left;">15분</td>
</tr>
<tr>
<td style="text-align: left;">3일 뒤</td>
<td style="text-align: left;">변형 문제 3개를 새로 풀기 · Pipeline 구조 그림을 가리고 직접 그리기</td>
<td style="text-align: left;">20분</td>
</tr>
<tr>
<td style="text-align: left;">7일 뒤</td>
<td style="text-align: left;">6장 <code>build()</code>를 보지 않고 다시 짜서 다른 표 데이터(펭귄 · island 열 포함)에 적용</td>
<td style="text-align: left;">30분</td>
</tr>
<tr>
<td style="text-align: left;">14일 뒤</td>
<td style="text-align: left;">오답 노트에 남은 문제만 다시 풀기 · 머신러닝-3 시작 전 선택표 · 3장 ✅ 줄만 읽기</td>
<td style="text-align: left;">15분</td>
</tr>
</tbody>
</table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 3개 · 전처리 노트북 + 로지스틱 회귀 Part 2(타이타닉 전처리) + 미션 3 정답1(Pipeline 부분)
- 중복 없이 5개 대단원 · 17개 섹션으로 재구성 · 6장 종합 실습(타이타닉 전처리 파이프라인)에서 연결
- 분할 · 교차검증은 [머신러닝-1](머신러닝-1_데이터분할·교차검증%28완성본%29.ipynb)을 그대로 씀 (설명 반복 없음)
- 딥러닝의 입력 정규화 · 임베딩으로 이어지는 기초

### 읽는 법

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">표시</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code> · <code>B</code></td>
<td style="text-align: left;"><strong>A</strong> 딥러닝 코드에 바로 나오는 필수 · <strong>B</strong> 알아두면 되는 것</td>
</tr>
<tr>
<td style="text-align: left;">🎯</td>
<td style="text-align: left;">이번 섹션에서 답해야 할 질문</td>
</tr>
<tr>
<td style="text-align: left;">💡</td>
<td style="text-align: left;">쉬운 설명 · 비유</td>
</tr>
<tr>
<td style="text-align: left;">⚠️</td>
<td style="text-align: left;">자주 하는 실수</td>
</tr>
<tr>
<td style="text-align: left;">✅</td>
<td style="text-align: left;">기억할 결론 = 🎯 질문의 답</td>
</tr>
</tbody>
</table>

**섹션 구성** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드 → 예측 정답 · 결과 해설 → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리

**코드 주석** · 코드 위 = 이 단계에서 하는 일, 코드 옆 = 실행 흐름 · 문법 구조

**셀 실행** · 위에서부터 차례로 (앞 섹션의 변수를 이어서 씀), 노트북과 같은 폴더의 `train.csv`(타이타닉)를 읽음

**머신러닝-1에서 가져다 쓰는 것** · `train_test_split(stratify=…)` · `StratifiedKFold(shuffle=True)` · `cross_val_score` · `GridSearchCV` · train · valid · test 3분할 · test는 1회만 → 이 노트북에서는 설명 없이 사용

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">원본</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">섹션</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">03 전처리 Part 1 셀 2 ~ 18</td>
<td style="text-align: left;">레이블 · 원-핫 · 오디널 인코딩, 가짜 순서</td>
<td style="text-align: left;">1-1 ~ 1-3 (<code>get_dummies</code> 비교는 추가)</td>
</tr>
<tr>
<td style="text-align: left;">03 셀 19 ~ 26</td>
<td style="text-align: left;">CountVectorizer 텍스트 인코딩</td>
<td style="text-align: left;">1-4</td>
</tr>
<tr>
<td style="text-align: left;">03 셀 27 ~ 37</td>
<td style="text-align: left;">펭귄 KNN 숙제 · Standard · Robust · MinMax</td>
<td style="text-align: left;">2-1 ~ 2-3</td>
</tr>
<tr>
<td style="text-align: left;">03 Part 3 셀 38 ~ 45</td>
<td style="text-align: left;">Normalizer · 코사인 유사도</td>
<td style="text-align: left;">2-4</td>
</tr>
<tr>
<td style="text-align: left;">03 Part 4 셀 46 ~ 63</td>
<td style="text-align: left;">fit / transform 규칙 · 누수 vs Pipeline · 난수 특성 선택</td>
<td style="text-align: left;">3-1 ~ 3-3</td>
</tr>
<tr>
<td style="text-align: left;">03 셀 52 · 64</td>
<td style="text-align: left;">Pipeline이란 · 오늘의 정리</td>
<td style="text-align: left;">4-1 · 10장</td>
</tr>
<tr>
<td style="text-align: left;">4-1 로지스틱 회귀 Part 2 (셀 6 ~ 9)</td>
<td style="text-align: left;">타이타닉 Cabin1 · 3분할 · ColumnTransformer(SimpleImputer + StandardScaler / OneHotEncoder)</td>
<td style="text-align: left;">4-2 · 4-3 (3분할 설명은 머신러닝-1 5-2 참조)</td>
</tr>
<tr>
<td style="text-align: left;">미션 3 정답1 셀 9 ~ 17</td>
<td style="text-align: left;">결측 확인 · 원-핫 · 거리 몫 · <code>preprocessing(scale_numeric)</code> · <code>model_pipe</code></td>
<td style="text-align: left;">2-1 (거리 몫) · 4-2 · 6장 (모델별 전처리 함수)</td>
</tr>
<tr>
<td style="text-align: left;">(추가)</td>
<td style="text-align: left;">전처리 설정 탐색, 이미지 mean · std, 임베딩</td>
<td style="text-align: left;">4-4 · 5-1 · 5-2</td>
</tr>
</tbody>
</table>

로지스틱 회귀 Part 1 · 3 ~ 6(시그모이드 · 확률 · 임곗값)과 미션 3의 모델 비교 · 앙상블은 머신러닝-3 범위라 제외. 원본 파일은 수정하지 않음.

### 전체 지도

#### 단원 연결 · 왜 이 순서로 배우는가

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: center;">순서</th>
<th style="text-align: left;">배우는 이유</th>
<th style="text-align: left;">대단원</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: center;">1</td>
<td style="text-align: left;">모델은 숫자만 읽으므로 글자부터 번역한다</td>
<td style="text-align: left;">인코딩</td>
</tr>
<tr>
<td style="text-align: center;">2</td>
<td style="text-align: left;">숫자가 되었어도 단위가 다르면 한 특성이 지배한다</td>
<td style="text-align: left;">스케일링 · 정규화</td>
</tr>
<tr>
<td style="text-align: center;">3</td>
<td style="text-align: left;">1 · 2의 기준을 어느 데이터로 만드느냐가 평가의 공정성을 정한다</td>
<td style="text-align: left;">fit · transform · 누수</td>
</tr>
<tr>
<td style="text-align: center;">4</td>
<td style="text-align: left;">1 ~ 3을 사람이 매번 지키기 어려우니 하나로 묶는다</td>
<td style="text-align: left;">파이프라인</td>
</tr>
<tr>
<td style="text-align: center;">5</td>
<td style="text-align: left;">같은 원리가 이미지 정규화 · 임베딩으로 이어진다</td>
<td style="text-align: left;">딥러닝 연결</td>
</tr>
<tr>
<td style="text-align: center;">6</td>
<td style="text-align: left;">1 ~ 5를 타이타닉 표 하나에 적용한다</td>
<td style="text-align: left;">종합 실습</td>
</tr>
</tbody>
</table>

#### ML/DL 공통 파이프라인 · 각 대단원이 쓰이는 곳

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰이는 장</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">① 데이터 준비</td>
<td style="text-align: left;">표 읽기 · 결측 확인 · 새 특성(Cabin1)</td>
<td style="text-align: left;">0장 · 4장</td>
</tr>
<tr>
<td style="text-align: left;">② 나누기</td>
<td style="text-align: left;">train · valid · test (머신러닝-1)</td>
<td style="text-align: left;">4장 · 6장</td>
</tr>
<tr>
<td style="text-align: left;">③ 전처리</td>
<td style="text-align: left;">인코딩 · 결측 채우기 · 스케일링 · 정규화 (train으로만 fit)</td>
<td style="text-align: left;"><strong>1장 ~ 5장</strong></td>
</tr>
<tr>
<td style="text-align: left;">④ 모델</td>
<td style="text-align: left;">전처리 + 모델 = Pipeline</td>
<td style="text-align: left;"><strong>4장</strong> · 6장</td>
</tr>
<tr>
<td style="text-align: left;">⑤ 학습 · 탐색</td>
<td style="text-align: left;">Pipeline째 교차검증 · GridSearchCV</td>
<td style="text-align: left;"><strong>3장</strong> · <strong>4장</strong></td>
</tr>
<tr>
<td style="text-align: left;">⑥ 평가</td>
<td style="text-align: left;">valid 1회 확인 (test는 머신러닝-3)</td>
<td style="text-align: left;">6장</td>
</tr>
</tbody>
</table>

**굵은 장** · 이 노트북의 핵심 = ③ 전처리를 ⑤ 교차검증 안에 넣어 누수를 막는 것

### 전체 코드

1 ~ 6장 핵심 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며(`train.csv`는 같은 폴더), 각 출력 앞 `[번호]`가 섹션.

In [23]:
# ---------- 0. 준비 ----------
import numpy as np                                            # 배열 계산
import pandas as pd                                           # 표 데이터
import seaborn as sns                                         # 펭귄 데이터
import torch                                                  # 임베딩 (5-2)
from sklearn.datasets import load_iris                        # 붓꽃 데이터
from sklearn.compose import ColumnTransformer                 # 열별 전처리
from sklearn.feature_extraction.text import CountVectorizer   # 문장 → 개수
from sklearn.feature_selection import SelectKBest, f_classif  # 열 고르기
from sklearn.impute import SimpleImputer                      # 빈칸 채우기
from sklearn.linear_model import LogisticRegression           # 선형(가중합) 모델
from sklearn.metrics.pairwise import cosine_similarity        # 방향 유사도
from sklearn.model_selection import (                         # 분할 · 교차검증 (머신러닝-1)
    train_test_split, StratifiedKFold, cross_val_score, GridSearchCV)
from sklearn.neighbors import KNeighborsClassifier            # 거리 모델 (스케일 영향 큼)
from sklearn.pipeline import Pipeline                         # 단계 묶음
from sklearn.preprocessing import (                           # 인코더 · 스케일러
    LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler,
    MinMaxScaler, RobustScaler, Normalizer)

SEED = 42                                                           # 원본 전처리 노트북과 같은 seed
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)  # 모든 비교에 같은 5등분
titanic = pd.read_csv("train.csv")                                  # 노트북 옆 파일
print("[0]", titanic.shape)                                         # 891 × 12

[0] (891, 12)


In [24]:
# ---------- 1. 인코딩 ----------
data = pd.DataFrame({"color": ["red", "green", "blue"] * 2})       # 색 6개
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")  # 모르는 값은 0
ohe.fit(data[["color"]])                                           # 범주 3개 기억
print("[1-1]", LabelEncoder().fit_transform(data["color"])[:3],    # 번호 · purple 원-핫
      ohe.transform(pd.DataFrame({"color": ["purple"]})).tolist())

np.random.seed(42)                                                          # 원본과 같은 색 100개
colors = pd.Series(np.random.choice(["red", "green", "blue"], size=100))    # 무작위 색
y_c = colors.map({"red": 1, "blue": 1, "green": 0})                         # 가운데 번호만 0인 규칙
X_label = LabelEncoder().fit_transform(colors).reshape(-1, 1)               # 1열 번호
X_onehot = OneHotEncoder(sparse_output=False).fit_transform(colors.to_frame())  # 3열 원-핫
print("[1-2]", LogisticRegression().fit(X_label, y_c).score(X_label, y_c),  # 0.64 vs 1.0
      LogisticRegression().fit(X_onehot, y_c).score(X_onehot, y_c))

order = [["small", "medium", "large"]]                                        # 지정 순서
enc = OrdinalEncoder(categories=order).fit_transform([["large"], ["small"]])  # 순서대로 번호
print("[1-3]", enc.ravel())                                                   # 2 · 0

vec = CountVectorizer().fit(["이 영화 정말 재밌다", "너무 지루하고 별로였다"])  # 사전 만들기
print("[1-4]", vec.transform(["이 영화는 별로다"]).nnz)                         # 사전 단어 0개

[1-1] [2 1 0] [[0.0, 0.0, 0.0]]
[1-2] 0.64 1.0
[1-3] [2. 0.]
[1-4] 0


In [25]:
# ---------- 2. 스케일링 · 정규화 ----------
X_hm = np.array([[150, 1000], [160, 20000], [170, 50000]], dtype=float)  # 키 · 용돈 3명
print("[2-1]", StandardScaler().fit_transform(X_hm)[:, 0].round(3))      # 키 변환

X_out = np.array([[1., 10.], [2., 12.], [3., 13.], [100., 1000.]])              # 마지막 행 이상치
for sc in [StandardScaler(), MinMaxScaler(), RobustScaler()]:                   # 스케일러 3종
    print("[2-2]", type(sc).__name__, sc.fit_transform(X_out)[:3, 0].round(3))  # 정상값 3개

meas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm",        # 측정값 4열
        "body_mass_g"]
peng = sns.load_dataset("penguins").dropna(subset=meas)                # 머신러닝-1과 같은 342마리
Xp, yp = peng[meas].to_numpy(), peng["species"].to_numpy()             # 표 → 배열
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(                         # 머신러닝-1과 같은 분할
    Xp, yp, test_size=0.2, stratify=yp, random_state=7)
cv_p = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)      # 같은 5등분
pipe_knn = Pipeline([("scaler", StandardScaler()),                     # 표준화 + KNN
                     ("knn", KNeighborsClassifier(n_neighbors=5))])
print("[2-3]", round(cross_val_score(pipe_knn, Xp_tr, yp_tr, cv=cv_p)  # 0.784 → 0.982
                     .mean(), 3))

A, B = np.array([[3., 4.]]), np.array([[8., 6.]])       # 방향 · 길이가 다른 두 벡터
print("[2-4]", Normalizer().fit_transform(A).round(3),  # 정규화 · 코사인
      round(float(cosine_similarity(A, B)[0, 0]), 4))

[2-1] [-1.225  0.     1.225]
[2-2] StandardScaler [-0.601 -0.577 -0.554]
[2-2] MinMaxScaler [0.   0.01 0.02]
[2-2] RobustScaler [-0.059 -0.02   0.02 ]
[2-3] 0.982
[2-4] [[0.6 0.8]] 0.96


In [26]:
# ---------- 3. fit · transform · 누수 ----------
X_iris, y_iris = load_iris(return_X_y=True)                              # 문제 · 정답만
X_tr, X_te, y_tr, y_te = train_test_split(                               # 원본과 같은 7 : 3
    X_iris, y_iris, test_size=0.3, stratify=y_iris, random_state=42)
sc = StandardScaler().fit(X_tr)                                          # 자는 train으로만
knn = KNeighborsClassifier(n_neighbors=5).fit(sc.transform(X_tr), y_tr)  # 변환 후 학습
print("[3-1]", round(knn.score(sc.transform(X_te), y_te), 3))            # 같은 자로 test

leak_iris = StandardScaler().fit_transform(X_iris)                     # 전체로 fit (누수)
knn5 = KNeighborsClassifier(n_neighbors=5)                             # 이웃 5개
pipe = Pipeline([("scaler", StandardScaler()), ("model", knn5)])       # fold마다 fit
print("[3-2]", round(cross_val_score(knn5, leak_iris, y_iris, cv=cv5)  # 누수 · Pipeline
                     .mean(), 3),
      round(cross_val_score(pipe, X_iris, y_iris, cv=cv5).mean(), 3))

rng = np.random.RandomState(42)                                       # 원본과 같은 난수
X_rand, y_rand = rng.normal(size=(100, 5000)), np.array([0, 1] * 50)  # 의미 없는 데이터
rng.shuffle(y_rand)                                                   # 정답 섞기
lr1000 = LogisticRegression(max_iter=1000)                            # 같은 모델
X_sel = SelectKBest(f_classif, k=20).fit_transform(X_rand, y_rand)    # 전체 정답을 보고 고름
pipe_sel = Pipeline([("select", SelectKBest(f_classif, k=20)),        # 고르기도 안으로
                     ("lr", lr1000)])
print("[3-3]", round(cross_val_score(lr1000, X_sel, y_rand, cv=cv5)   # 0.87 vs 0.52
                     .mean(), 3),
      round(cross_val_score(pipe_sel, X_rand, y_rand, cv=cv5).mean(), 3))

[3-1] 0.911
[3-2] 0.967 0.973


[3-3] 0.87 0.52


In [27]:
# ---------- 4. 파이프라인 (타이타닉) ----------
X_t = titanic.drop(columns="Survived").copy()                       # 정답 열 빼기
X_t["Cabin1"] = X_t["Cabin"].fillna("N").str[:1]                    # 객실 첫 글자
X_t = X_t.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])  # 안 쓸 열 제거
y_t = titanic["Survived"]                                           # 정답
X_dev, X_test, y_dev, y_test = train_test_split(                    # test 20% 봉인
    X_t, y_t, test_size=0.2, stratify=y_t, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(              # 나머지를 train · valid로
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=42)
num_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare"]              # 숫자 5열
cat_cols = ["Sex", "Embarked", "Cabin1"]                            # 글자 3열
print("[4-2]", SimpleImputer(strategy="median")                     # train 중앙값
      .fit(X_train[["Age"]]).statistics_)

num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),         # 숫자 라인
                     ("scaler", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),  # 글자 라인
                     ("onehot", OneHotEncoder(handle_unknown="ignore"))])
prep = ColumnTransformer([("num", num_pipe, num_cols),                      # 두 라인을 한 묶음으로
                          ("cat", cat_pipe, cat_cols)])
print("[4-3]", prep.fit_transform(X_train).shape)                           # 19열

lr_pipe = Pipeline([("prep", prep),                                       # 전처리 + 로지스틱
                    ("model", LogisticRegression(max_iter=5000))])
grid_t = GridSearchCV(lr_pipe, {"model__C": [0.01, 0.1, 1, 10]}, cv=cv5)  # 규제 세기 탐색
grid_t.fit(X_train, y_train)                                              # train 안에서만
print("[4-4]", grid_t.best_params_, round(grid_t.best_score_, 3))         # 고른 C

[4-2]

 [29.]
[4-3] (534, 19)


[4-4] {'model__C': 0.1} 0.815


In [28]:
# ---------- 5. 딥러닝 연결 ----------
rng_img = np.random.default_rng(0)                                              # 가짜 사진 고정
img_train = rng_img.integers(0, 256, size=(20, 8, 8, 3), dtype=np.uint8) / 255  # 0 ~ 1 사진
mean, std = img_train.mean(axis=(0, 1, 2)), img_train.std(axis=(0, 1, 2))       # train 채널 통계
print("[5-1]", mean.round(3), std.round(3))                                     # 저장할 값

vocab = {"<unk>": 0, "C": 1, "Q": 2, "S": 3}                    # train 사전
emb = torch.nn.Embedding(len(vocab), 3)                         # 번호마다 3칸
idx = torch.tensor([vocab.get(p, 0) for p in ["S", "C", "X"]])  # 모르는 값은 0
print("[5-2]", idx.tolist(), tuple(emb(idx).shape))             # 번호 · 벡터 모양

[5-1] [0.497 0.506 0.49 ] [0.289 0.29  0.302]
[5-2] [3, 1, 0] (3, 3)


In [29]:
# ---------- 6. 종합 실습 ----------
final_t = Pipeline([("prep", prep),                      # 전처리 + 로지스틱
                    ("model", LogisticRegression(max_iter=5000))])
final_t.fit(X_train, y_train)                            # train 전체로
print("[6]", round(final_t.score(X_valid, y_valid), 3))  # valid 1회

[6] 0.798
